In [ ]:
# UHViT: Complete Training Pipeline for IF 5+ Journal
## Skin Lesion Classification with Uncertainty Quantification


In [ ]:
#Install Dependencies (Fixed for RunPod)

# Upgrade all packages to numpy 2.x compatible versions
!pip install -q --upgrade numpy==2.0.2
!pip install -q --upgrade matplotlib>=3.8.0 scipy>=1.12.0 scikit-learn>=1.4.0
!pip install -q --upgrade timm>=1.0.0
!pip install -q --upgrade wandb>=0.17.0  # Fixed: needs numpy 2.x compatible version

# Now install the rest
!pip install -q albumentations>=1.3.1
!pip install -q kaggle==1.5.16
!pip install -q grad-cam
!pip install -q pandas>=2.0.3
!pip install -q seaborn>=0.13.0
!pip install -q tqdm>=4.66.1
!pip install -q ptflops>=0.7
!pip install -q gdown
!pip install -q opencv-python-headless


In [2]:
# CELL 1B: EXTRA DEPENDENCY FOR EXTERNAL DATASET
!pip install -q medmnist
print("✓ medmnist installed")

✓ medmnist installed


In [3]:
# CELL 2: IMPORTS
import os, sys, json, random, time, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
import timm
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_auc_score, precision_recall_fscore_support, roc_curve, auc
from sklearn.calibration import calibration_curve
from scipy import stats
import wandb
warnings.filterwarnings('ignore')
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")



PyTorch: 2.8.0+cu128, CUDA: True
GPU: NVIDIA GeForce RTX 5090


In [4]:
# CELL 3 — CONFIGURATION (REVISED)
import os
import random
import numpy as np
import torch
from pathlib import Path

CONFIG = {
    # Hard-coded keys (as requested)
    "KAGGLE_USERNAME": "sushovanchaudhury",  # UPDATE THIS if needed
    "KAGGLE_KEY": "f348cdc3864d8b2ac8282969811b3700",  # UPDATE THIS if needed
    "WANDB_API_KEY": "07ac9d1305afd5973d9022f150eb134c3b77ca7b",  # UPDATE THIS if needed

    "seed": 42,
    "n_folds": 5,
    "epochs": 30,
    "batch_size": 32,
    "img_size": 224,
    "num_workers": 4,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "mc_dropout_samples": 10,
    "num_classes": 8,
    "dropout_rate": 0.3,
    "swin_model": "swin_tiny_patch4_window7_224",
    "efficientnet_model": "efficientnet_b3",
    "class_names": ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"],
    "data_dir": "/workspace/data",
    "output_dir": "/workspace/outputs",
    "checkpoint_dir": "/workspace/checkpoints",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

for d in [CONFIG["data_dir"], CONFIG["output_dir"], CONFIG["checkpoint_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(CONFIG["seed"])
print(f"Config set. Device: {CONFIG['device']}")

Config set. Device: cuda


In [5]:
# CELL 4 — SETUP APIs (REVISED, FAULT-TOLERANT)
import os
import json
import wandb

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": CONFIG["KAGGLE_USERNAME"], "key": CONFIG["KAGGLE_KEY"]}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

os.environ["WANDB_API_KEY"] = CONFIG["WANDB_API_KEY"]
WANDB_ENABLED = True
try:
    wandb.login(key=CONFIG["WANDB_API_KEY"])
    print("APIs configured (W&B login OK)")
except Exception as e:
    WANDB_ENABLED = False
    os.environ["WANDB_MODE"] = "disabled"
    print(f"⚠ W&B login failed ({e}). Continuing with WANDB_MODE=disabled — training will still run, just without W&B logging.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sushovan-chaudhury (sushovan-chaudhury-bits-pilani) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


APIs configured (W&B login OK)


In [7]:
# CELL 5: DOWNLOAD + LOCATE DATASETS (ISIC2019 + HAM10000) — DE-DUPLICATED, FAULT-TOLERANT
from pathlib import Path
import pandas as pd

def download_and_extract(kaggle_slug, extract_subdir, zip_name=None):
    """Idempotent Kaggle dataset download+extract. Skips if target dir already has files."""
    target_dir = Path(CONFIG['data_dir']) / extract_subdir
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f"✓ {extract_subdir} already present, skipping download")
        return target_dir
    target_dir.mkdir(parents=True, exist_ok=True)
    zip_file = zip_name or (kaggle_slug.split('/')[-1] + '.zip')
    zip_path = Path(CONFIG['data_dir']) / zip_file

    if not zip_path.exists():
        get_ipython().system(f"kaggle datasets download -d {kaggle_slug} -p {CONFIG['data_dir']}")

    if not zip_path.exists():
        raise RuntimeError(
            f"Kaggle download failed for '{kaggle_slug}': no zip file found at {zip_path}.\n"
            f"Check your KAGGLE_USERNAME / KAGGLE_KEY in CONFIG, your Kaggle account's dataset access, "
            f"and network connectivity, then re-run this cell."
        )

    get_ipython().system(f"unzip -q {zip_path} -d {target_dir}")

    if not any(target_dir.iterdir()):
        raise RuntimeError(
            f"Unzip produced no files in {target_dir}. The downloaded file at {zip_path} may be "
            f"corrupted, an HTML error page, or incomplete. Delete it and re-run this cell."
        )

    zip_path.unlink(missing_ok=True)
    return target_dir

def find_label_csv(base_dir, must_have_cols=None, prefer_keywords=('ground', 'truth')):
    """Find the correct label CSV: prefer files matching keywords AND containing required columns."""
    csv_candidates = list(base_dir.rglob("*.csv"))
    if not csv_candidates:
        raise FileNotFoundError(f"No CSV files found under {base_dir}. The dataset may not have extracted correctly.")

    def score(p):
        name = p.name.lower()
        keyword_hit = any(k in name for k in prefer_keywords)
        has_cols = True
        if must_have_cols:
            try:
                cols = pd.read_csv(p, nrows=5).columns
                has_cols = all(c in cols for c in must_have_cols)
            except Exception:
                has_cols = False
        # Prefer: keyword match AND required columns present, then keyword match, then shortest name
        return (not (keyword_hit and has_cols), not keyword_hit, len(name))

    best_csv = sorted(csv_candidates, key=score)[0]
    return best_csv

def find_image_dir(base_dir):
    """Find the directory with the most image files under base_dir."""
    candidate_dirs = [d for d in base_dir.rglob("*") if d.is_dir()]
    candidate_dirs.append(base_dir)

    def image_count(d):
        return sum(len(list(d.glob(ext))) for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.png"))

    best_dir = max(candidate_dirs, key=image_count)
    if image_count(best_dir) == 0:
        raise FileNotFoundError(f"No image files (.jpg/.png) found anywhere under {base_dir}.")
    return best_dir


# ---- ISIC2019 ----
isic_base = download_and_extract("andrewmvd/isic-2019", "isic-2019")

train_csv = find_label_csv(isic_base, must_have_cols=CONFIG["class_names"], prefer_keywords=("ground", "truth"))
train_df = pd.read_csv(train_csv)
train_img_dir = find_image_dir(isic_base)

missing_cols = [c for c in CONFIG["class_names"] if c not in train_df.columns]
if missing_cols:
    raise ValueError(
        f"ISIC label CSV missing required class columns: {missing_cols}\n"
        f"CSV path used: {train_csv}\n"
        f"Loaded columns (first 20): {list(train_df.columns[:20])}"
    )

print("✓ ISIC2019 label CSV:", train_csv, "| size:", len(train_df))
print("✓ ISIC2019 image dir:", train_img_dir, "| image files:", sum(
    len(list(train_img_dir.glob(ext))) for ext in ("*.jpg", "*.JPG", "*.jpeg", "*.png")))
print("Class distribution (sum of one-hot columns):")
for cls in CONFIG["class_names"]:
    print(f"  {cls}: {int(train_df[cls].sum())}")

# ---- HAM10000 ----
ham_base = download_and_extract("kmader/skin-cancer-mnist-ham10000", "ham10000")
ham_csv = find_label_csv(ham_base, must_have_cols=["dx", "image_id"], prefer_keywords=("metadata",))
ham_img_dirs = [d for d in ham_base.iterdir() if d.is_dir() and "HAM" in d.name.upper()] or [ham_base]

print("✓ HAM10000 label CSV:", ham_csv, "| size:", len(pd.read_csv(ham_csv)))
print("✓ HAM10000 image dir(s):", ham_img_dirs)

print("\n✓ Datasets ready (ISIC2019, HAM10000) — no redundant re-scanning")


✓ isic-2019 already present, skipping download
✓ ISIC2019 label CSV: /workspace/data/isic-2019/ISIC_2019_Training_GroundTruth.csv | size: 25331
✓ ISIC2019 image dir: /workspace/data/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input | image files: 25331
Class distribution (sum of one-hot columns):
  AK: 867
  BCC: 3323
  BKL: 2624
  DF: 239
  MEL: 4522
  NV: 12875
  SCC: 628
  VASC: 253
✓ ham10000 already present, skipping download
✓ HAM10000 label CSV: /workspace/data/ham10000/HAM10000_metadata.csv | size: 10015
✓ HAM10000 image dir(s): [PosixPath('/workspace/data/ham10000/ham10000_images_part_2'), PosixPath('/workspace/data/ham10000/ham10000_images_part_1'), PosixPath('/workspace/data/ham10000/HAM10000_images_part_2'), PosixPath('/workspace/data/ham10000/HAM10000_images_part_1')]

✓ Datasets ready (ISIC2019, HAM10000) — no redundant re-scanning


In [8]:
def get_train_transforms(img_size):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=45, p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10, 50)),
            A.GaussianBlur(blur_limit=(3, 7))
        ], p=0.3),
        A.OneOf([
            A.CLAHE(clip_limit=4.0),
            A.RandomBrightnessContrast()
        ], p=0.5),
        A.HueSaturationValue(p=0.5),
        A.CoarseDropout(
            num_holes_range=(1, 8),
            hole_height_range=(img_size//16, img_size//8),
            hole_width_range=(img_size//16, img_size//8),
            p=0.3
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])


In [9]:
import albumentations as A
import albumentations
print("Albumentations version:", albumentations.__version__)

# instantiate to force validation now
t = get_train_transforms(CONFIG["img_size"])
print("Train transforms OK:", t)

Albumentations version: 2.0.8
Train transforms OK: Compose([
  RandomResizedCrop(p=1.0, area_for_downscale=None, interpolation=1, mask_interpolation=0, ratio=(0.75, 1.3333333333333333), scale=(0.8, 1.0), size=(224, 224)),
  HorizontalFlip(p=0.5),
  VerticalFlip(p=0.5),
  RandomRotate90(p=0.5),
  ShiftScaleRotate(p=0.5, shift_limit_x=(-0.1, 0.1), shift_limit_y=(-0.1, 0.1), scale_limit=(-0.09999999999999998, 0.10000000000000009), rotate_limit=(-45.0, 45.0), interpolation=1, border_mode=0, fill=0.0, fill_mask=0.0, rotate_method='largest_box', mask_interpolation=0),
  OneOf([
    GaussNoise(p=0.5, mean_range=(0.0, 0.0), noise_scale_factor=1.0, per_channel=True, std_range=(0.2, 0.44)),
    GaussianBlur(p=0.5, blur_limit=(3, 7), sigma_limit=(0.5, 3.0)),
  ], p=0.3),
  OneOf([
    CLAHE(p=0.5, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8)),
    RandomBrightnessContrast(p=0.5, brightness_by_max=True, brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), ensure_safe_range=False),
  ], p=0.5)

In [10]:
# CELL 7: DATASET CLASS (HARDENED)
class ISIC2019Dataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['image']
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            p = self.img_dir / f"{img_name}{ext}"
            if p.exists(): img_path = p; break
        if not img_path:
            for f in self.img_dir.glob(f"{img_name}*"): img_path = f; break
        if img_path is None:
            raise FileNotFoundError(
                f"ISIC2019Dataset: could not find image for '{img_name}' under {self.img_dir}. "
                f"Check that train_img_dir points to the correct extracted image folder."
            )
        image = np.array(Image.open(img_path).convert('RGB'))
        label = 0
        for i, cls in enumerate(CONFIG['class_names']):
            if cls in row and row[cls] == 1.0: label = i; break
        if self.transform: image = self.transform(image=image)['image']
        return image, label

class HAM10000Dataset(Dataset):
    def __init__(self, csv_path, img_dirs, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dirs = [Path(d) for d in img_dirs] if isinstance(img_dirs, list) else [Path(img_dirs)]
        self.transform = transform
        self.label_map = {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 7}
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = None
        for d in self.img_dirs:
            for ext in ['.jpg', '.jpeg', '.png']:
                p = d / f"{row['image_id']}{ext}"
                if p.exists(): img_path = p; break
            if img_path: break
        if img_path is None:
            raise FileNotFoundError(
                f"HAM10000Dataset: could not find image for '{row['image_id']}' under {self.img_dirs}. "
                f"Check that ham_img_dirs points to the correct extracted image folder(s)."
            )
        image = np.array(Image.open(img_path).convert('RGB'))
        label = self.label_map.get(row['dx'], 0)
        if self.transform: image = self.transform(image=image)['image']
        return image, label
print("✓ Dataset classes defined")


✓ Dataset classes defined


In [11]:
# CELL 8: VERIFY LOADED DATA (train_df / train_img_dir set in CELL 5)
if "train_df" not in globals() or "train_img_dir" not in globals():
    raise RuntimeError("train_df / train_img_dir not found. Run CELL 5 (download+locate) first.")

print("train_df size:", len(train_df))
print("train_img_dir:", train_img_dir)
print("Class columns present:", [c for c in CONFIG["class_names"] if c in train_df.columns])


train_df size: 25331
train_img_dir: /workspace/data/isic-2019/ISIC_2019_Training_Input/ISIC_2019_Training_Input
Class columns present: ['AK', 'BCC', 'BKL', 'DF', 'MEL', 'NV', 'SCC', 'VASC']


In [12]:
ds = ISIC2019Dataset(train_df.iloc[:200], train_img_dir, get_train_transforms(CONFIG["img_size"]))
x, y = ds[0]
print("Sample tensor:", x.shape, "label:", y)
print("Unique labels in first 200:", sorted({ds[i][1] for i in range(200)}))

Sample tensor: torch.Size([3, 224, 224]) label: 5
Unique labels in first 200: [4, 5]


In [13]:
# Build labels exactly like your dataset does
labels = []
for i in range(len(train_df)):
    row = train_df.iloc[i]
    found = False
    for j, cls in enumerate(CONFIG["class_names"]):
        if cls in train_df.columns and row[cls] == 1.0:
            labels.append(j)
            found = True
            break
    if not found:
        labels.append(-1)

labels = np.array(labels)
print("Label counts:", np.bincount(labels[labels>=0], minlength=CONFIG["num_classes"]))
print("Missing labels (-1):", np.sum(labels == -1))
print("Unique labels:", np.unique(labels))

Label counts: [  867  3323  2624   239  4522 12875   628   253]
Missing labels (-1): 0
Unique labels: [0 1 2 3 4 5 6 7]


In [14]:
# CELL 9: UHViT MODEL
class GatedFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(dim * 2, dim), nn.Sigmoid())
        self.proj = nn.Linear(dim * 2, dim)
    def forward(self, x1, x2):
        concat = torch.cat([x1, x2], dim=-1)
        return self.proj(concat) + self.gate(concat) * x1 + (1 - self.gate(concat)) * x2

class MCDropout(nn.Module):
    def __init__(self, p=0.3): super().__init__(); self.p = p
    def forward(self, x): return F.dropout(x, p=self.p, training=True)

class UHViT(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super().__init__()
        self.swin = timm.create_model(CONFIG['swin_model'], pretrained=True, num_classes=0)
        self.efficientnet = timm.create_model(CONFIG['efficientnet_model'], pretrained=True, num_classes=0)
        self.swin_proj = nn.Linear(self.swin.num_features, 512)
        self.eff_proj = nn.Linear(self.efficientnet.num_features, 512)
        self.fusion = GatedFusion(512)
        self.mc_dropout = MCDropout(dropout_rate)
        self.classifier = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, 256), nn.GELU(), MCDropout(dropout_rate), nn.Linear(256, num_classes))
    def forward(self, x):
        swin_feat = self.swin_proj(self.swin(x))
        eff_feat = self.eff_proj(self.efficientnet(x))
        return self.classifier(self.mc_dropout(self.fusion(swin_feat, eff_feat)))
model = UHViT(); print(f"✓ UHViT: {sum(p.numel() for p in model.parameters()):,} params"); del model



model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

✓ UHViT: 40,580,266 params


In [15]:
# NEW CELL 9B — GATE-VALUE LOGGING WRAPPER
import torch.nn as nn
import timm

class UHViTWithGateLogging(nn.Module):
    """Wraps UHViT to expose gate values g during forward pass."""
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super().__init__()
        self.swin = timm.create_model(CONFIG["swin_model"], pretrained=True, num_classes=0)
        self.efficientnet = timm.create_model(CONFIG["efficientnet_model"], pretrained=True, num_classes=0)
        self.swin_proj = nn.Linear(self.swin.num_features, 512)
        self.eff_proj = nn.Linear(self.efficientnet.num_features, 512)
        self.fusion = GatedFusion(512)
        self.mc_dropout = MCDropout(dropout_rate)
        self.classifier = nn.Sequential(
            nn.LayerNorm(512),
            nn.Linear(512, 256),
            nn.GELU(),
            MCDropout(dropout_rate),
            nn.Linear(256, num_classes),
        )
        self.last_gate = None

    def forward(self, x):
        swin_feat = self.swin_proj(self.swin(x))
        eff_feat = self.eff_proj(self.efficientnet(x))
        concat = torch.cat([swin_feat, eff_feat], dim=-1)
        gate = self.fusion.gate(concat)
        self.last_gate = gate.detach().mean(dim=1).cpu().numpy()
        fused = self.fusion.proj(concat) + gate * swin_feat + (1 - gate) * eff_feat
        return self.classifier(self.mc_dropout(fused))

print("UHViTWithGateLogging defined")

UHViTWithGateLogging defined


In [16]:
# CELL 10: ABLATION & SOTA MODELS
class SwinOnly(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super().__init__()
        self.swin = timm.create_model(CONFIG['swin_model'], pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(nn.LayerNorm(self.swin.num_features), nn.Dropout(dropout_rate), nn.Linear(self.swin.num_features, num_classes))
    def forward(self, x): return self.classifier(self.swin(x))

class EfficientNetOnly(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super().__init__()
        self.eff = timm.create_model(CONFIG['efficientnet_model'], pretrained=True, num_classes=0)
        self.classifier = nn.Sequential(nn.LayerNorm(self.eff.num_features), nn.Dropout(dropout_rate), nn.Linear(self.eff.num_features, num_classes))
    def forward(self, x): return self.classifier(self.eff(x))

class UHViTConcatFusion(nn.Module):
    def __init__(self, num_classes=8, dropout_rate=0.3):
        super().__init__()
        self.swin = timm.create_model(CONFIG['swin_model'], pretrained=True, num_classes=0)
        self.efficientnet = timm.create_model(CONFIG['efficientnet_model'], pretrained=True, num_classes=0)
        self.swin_proj = nn.Linear(self.swin.num_features, 512)
        self.eff_proj = nn.Linear(self.efficientnet.num_features, 512)
        self.classifier = nn.Sequential(nn.LayerNorm(1024), nn.Dropout(dropout_rate), nn.Linear(1024, num_classes))
    def forward(self, x): return self.classifier(torch.cat([self.swin_proj(self.swin(x)), self.eff_proj(self.efficientnet(x))], dim=-1))

SOTA_MODELS = ['resnet50', 'efficientnet_b3', 'vit_base_patch16_224', 'swin_tiny_patch4_window7_224', 'convnext_tiny', 'densenet121']
def create_sota_model(name, num_classes=8): return timm.create_model(name, pretrained=True, num_classes=num_classes)
print("✓ Ablation & SOTA models defined")



✓ Ablation & SOTA models defined


In [17]:
# CELL 11: LOSS FUNCTIONS
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0): super().__init__(); self.alpha = alpha; self.gamma = gamma
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

class ClassBalancedLoss(nn.Module):
    def __init__(self, samples_per_class, num_classes, beta=0.9999, gamma=2.0):
        super().__init__()
        effective_num = 1.0 - np.power(beta, samples_per_class)
        weights = (1.0 - beta) / np.array(effective_num)
        self.weights = torch.tensor(weights / np.sum(weights) * num_classes, dtype=torch.float32)
        self.gamma = gamma
    def forward(self, inputs, targets):
        self.weights = self.weights.to(inputs.device)
        ce = F.cross_entropy(inputs, targets, weight=self.weights, reduction='none')
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

def get_class_weights(labels):
    counts = np.maximum(np.bincount(labels, minlength=CONFIG['num_classes']), 1)
    weights = 1.0 / counts
    return torch.tensor(weights / weights.sum() * len(weights), dtype=torch.float32)

def create_balanced_sampler(labels):
    counts = np.maximum(np.bincount(labels, minlength=CONFIG['num_classes']), 1)
    sample_weights = (1.0 / counts)[labels]
    return WeightedRandomSampler(sample_weights, len(labels), replacement=True)
print("✓ Loss functions defined")



✓ Loss functions defined


In [18]:
# CELL 12: TRAINING FUNCTIONS
def train_epoch(model, loader, criterion, optimizer, scheduler, device, epoch):
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []
    pbar = tqdm(loader, desc=f'Train E{epoch}')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        running_loss += loss.item()
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return running_loss / len(loader), accuracy_score(all_labels, all_preds), balanced_accuracy_score(all_labels, all_preds)

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Val'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            running_loss += criterion(outputs, labels).item()
            all_probs.extend(F.softmax(outputs, dim=1).cpu().numpy())
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_preds, all_labels, all_probs = np.array(all_preds), np.array(all_labels), np.array(all_probs)
    return running_loss / len(loader), accuracy_score(all_labels, all_preds), balanced_accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels, all_probs

def validate_with_tta(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='TTA'):
            images = images.to(device)
            probs = torch.stack([F.softmax(model(images), dim=1), F.softmax(model(torch.flip(images, [3])), dim=1), F.softmax(model(torch.flip(images, [2])), dim=1)]).mean(0)
            all_probs.extend(probs.cpu().numpy()); all_labels.extend(labels.numpy())
    all_probs, all_labels = np.array(all_probs), np.array(all_labels)
    all_preds = all_probs.argmax(axis=1)
    return accuracy_score(all_labels, all_preds), balanced_accuracy_score(all_labels, all_preds), f1_score(all_labels, all_preds, average='macro'), all_preds, all_labels, all_probs
print("✓ Training functions defined")



✓ Training functions defined


In [19]:
# CELL 13 — UNCERTAINTY ESTIMATION (REVISED)
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import numpy as np

def enable_mc_dropout_only(module):
    classname = module.__class__.__name__
    if isinstance(module, (nn.Dropout, MCDropout)):
        module.train()
    elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
        module.eval()

def estimate_uncertainty(model, loader, device, n_samples=10):
    model.eval()
    model.apply(enable_mc_dropout_only)

    all_means, all_stds, all_entropy, all_labels = [], [], [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=f"Uncertainty T={n_samples}"):
            images = images.to(device)
            outputs = torch.stack(
                [F.softmax(model(images), dim=1) for _ in range(n_samples)], dim=0
            )

            mean_probs = outputs.mean(0)
            std_probs = outputs.std(0).mean(1)
            entropy = -(mean_probs.clamp_min(1e-9) * mean_probs.clamp_min(1e-9).log()).sum(1)

            all_means.extend(mean_probs.cpu().numpy())
            all_stds.extend(std_probs.cpu().numpy())
            all_entropy.extend(entropy.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_means), np.array(all_stds), np.array(all_entropy), np.array(all_labels)

print("Uncertainty functions defined (revised)")


Uncertainty functions defined (revised)


In [20]:
# CELL 14: VISUALIZATION FUNCTIONS
def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('Normalized'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); wandb.log({"confusion_matrix": wandb.Image(save_path)}); plt.close()

def plot_roc_curves(y_true, y_probs, class_names, save_path):
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, cls in enumerate(class_names):
        binary = (y_true == i).astype(int)
        if 0 < binary.sum() < len(binary):
            fpr, tpr, _ = roc_curve(binary, y_probs[:, i])
            ax.plot(fpr, tpr, label=f'{cls} (AUC={auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], 'k--'); ax.legend(loc='lower right')
    plt.savefig(save_path, dpi=150); wandb.log({"roc_curves": wandb.Image(save_path)}); plt.close()

def plot_training_history(history, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    epochs = range(1, len(history['train_loss']) + 1)
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train'); axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val'); axes[0, 0].set_title('Loss'); axes[0, 0].legend()
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train'); axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val'); axes[0, 1].set_title('Accuracy'); axes[0, 1].legend()
    axes[1, 0].plot(epochs, history['train_bacc'], 'b-', label='Train'); axes[1, 0].plot(epochs, history['val_bacc'], 'r-', label='Val'); axes[1, 0].set_title('Balanced Accuracy'); axes[1, 0].legend()
    axes[1, 1].plot(epochs, history['val_f1'], 'g-'); axes[1, 1].set_title('Macro F1')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); wandb.log({"training_history": wandb.Image(save_path)}); plt.close()

def plot_uncertainty_analysis(uncertainty, preds, labels, save_path):
    """Visualize MC-dropout uncertainty: distribution, correctness split, and per-class summary.
    uncertainty: array of per-sample std-dev (or entropy) scores
    preds: predicted class indices
    labels: true class indices
    """
    uncertainty = np.asarray(uncertainty)
    preds = np.asarray(preds)
    labels = np.asarray(labels)
    is_correct = (preds == labels)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].hist(uncertainty, bins=30, color='steelblue', alpha=0.8)
    axes[0].set_title('Uncertainty Distribution')
    axes[0].set_xlabel('Uncertainty (std)')
    axes[0].set_ylabel('Count')

    if is_correct.any() and (~is_correct).any():
        axes[1].hist(uncertainty[is_correct], bins=30, alpha=0.6, label='Correct', color='seagreen')
        axes[1].hist(uncertainty[~is_correct], bins=30, alpha=0.6, label='Incorrect', color='indianred')
        axes[1].legend()
    else:
        axes[1].hist(uncertainty, bins=30, color='gray', alpha=0.8)
    axes[1].set_title('Uncertainty: Correct vs Incorrect')
    axes[1].set_xlabel('Uncertainty (std)')
    axes[1].set_ylabel('Count')

    mean_correct = uncertainty[is_correct].mean() if is_correct.any() else 0.0
    mean_incorrect = uncertainty[~is_correct].mean() if (~is_correct).any() else 0.0
    axes[2].bar(['Correct', 'Incorrect'], [mean_correct, mean_incorrect], color=['seagreen', 'indianred'])
    axes[2].set_title('Mean Uncertainty by Correctness')
    axes[2].set_ylabel('Mean uncertainty')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    if wandb.run is not None:
        wandb.log({"uncertainty_analysis": wandb.Image(save_path)})
    plt.close()

print("✓ Visualization functions defined")


✓ Visualization functions defined


In [21]:
# CELL 15: CLINICAL & CALIBRATION ANALYSIS
def compute_ece(probs, labels, n_bins=15):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs > bin_boundaries[i]) & (probs <= bin_boundaries[i + 1])
        if mask.sum() > 0: ece += mask.sum() * abs(probs[mask].mean() - labels[mask].mean())
    return ece / len(probs)

def per_class_analysis(y_true, y_pred, y_probs, class_names):
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    auc_scores = []
    for i in range(len(class_names)):
        binary = (y_true == i).astype(int)
        try: auc_scores.append(roc_auc_score(binary, y_probs[:, i]) if 0 < binary.sum() < len(binary) else 0)
        except: auc_scores.append(0)
    df = pd.DataFrame({'Class': class_names, 'Precision': precision, 'Recall': recall, 'F1': f1, 'AUC': auc_scores, 'Support': support})
    wandb.log({"per_class_metrics": wandb.Table(dataframe=df)})
    return df

def plot_per_class_performance(df, save_path):
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(df)); width = 0.2
    ax.bar(x - 1.5*width, df['Precision'], width, label='Precision'); ax.bar(x - 0.5*width, df['Recall'], width, label='Recall')
    ax.bar(x + 0.5*width, df['F1'], width, label='F1'); ax.bar(x + 1.5*width, df['AUC'], width, label='AUC')
    ax.set_xticks(x); ax.set_xticklabels(df['Class'], rotation=45); ax.legend(); ax.set_ylim(0, 1)
    plt.tight_layout(); plt.savefig(save_path, dpi=150); wandb.log({"per_class_chart": wandb.Image(save_path)}); plt.close()
print("✓ Clinical analysis functions defined")



✓ Clinical analysis functions defined


In [22]:
# CELL 16: GRAD-CAM++
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def get_gradcam_visualizations(model, loader, device, save_path='gradcam.png'):
    model.eval()
    try: target_layer = [model.efficientnet.conv_head]
    except: target_layer = [list(model.efficientnet.children())[-2]]
    cam = GradCAMPlusPlus(model=model, target_layers=target_layer)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.flatten()
    images, labels = next(iter(loader)); images = images[:8].to(device); labels = labels[:8]
    for i in range(min(8, len(images))):
        input_tensor = images[i:i + 1]
        with torch.no_grad(): pred_class = model(input_tensor).argmax(dim=1).item()
        grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_class)])[0, :]
        img = images[i].cpu().numpy().transpose(1, 2, 0)
        img = np.clip(img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)
        vis = show_cam_on_image(img, grayscale_cam, use_rgb=True)
        axes[i].imshow(vis); axes[i].set_title(f'T:{CONFIG["class_names"][labels[i]]} P:{CONFIG["class_names"][pred_class]}'); axes[i].axis('off')
    plt.tight_layout(); plt.savefig(save_path, dpi=150); wandb.log({"gradcam": wandb.Image(save_path)}); plt.close(); del cam
print("✓ Grad-CAM++ defined")



✓ Grad-CAM++ defined


In [23]:
# CELL 17: STATISTICAL TESTS
def mcnemar_test(y_true, y_pred1, y_pred2):
    c1, c2 = (y_pred1 == y_true), (y_pred2 == y_true)
    b, c = np.sum(c1 & ~c2), np.sum(~c1 & c2)
    if b + c > 0: stat = (abs(b - c) - 1) ** 2 / (b + c); p = 1 - stats.chi2.cdf(stat, df=1)
    else: stat, p = 0, 1.0
    return stat, p

def paired_t_test(scores1, scores2): return stats.ttest_rel(scores1, scores2)
def wilcoxon_test(scores1, scores2):
    try: return stats.wilcoxon(scores1, scores2)
    except: return 0, 1.0

def compute_ci(scores, confidence=0.95):
    n, mean, se = len(scores), np.mean(scores), stats.sem(scores)
    h = se * stats.t.ppf((1 + confidence) / 2, n - 1)
    return mean, mean - h, mean + h
print("✓ Statistical tests defined")



✓ Statistical tests defined


In [24]:
# CELL 18: COMPUTATIONAL ANALYSIS
from ptflops import get_model_complexity_info

def compute_model_complexity(model, input_size=(3, 224, 224)):
    try: macs, params = get_model_complexity_info(model, input_size, as_strings=False, print_per_layer_stat=False, verbose=False); return macs * 2, params
    except: return 0, sum(p.numel() for p in model.parameters())

def measure_inference_time(model, device, n_runs=100):
    model.eval(); dummy = torch.randn(1, 3, 224, 224).to(device)
    with torch.no_grad():
        for _ in range(10): _ = model(dummy)
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            start = time.time(); _ = model(dummy)
            if device == 'cuda': torch.cuda.synchronize()
            times.append(time.time() - start)
    return np.mean(times) * 1000, np.std(times) * 1000

def computational_analysis(models_dict, device):
    results = []
    for name, model_fn in models_dict.items():
        print(f"Analyzing {name}..."); model = model_fn().to(device)
        flops, params = compute_model_complexity(model); mean_time, std_time = measure_inference_time(model, device)
        results.append({'Model': name, 'Params (M)': params / 1e6, 'FLOPs (G)': flops / 1e9, 'Time (ms)': mean_time})
        del model; torch.cuda.empty_cache()
    df = pd.DataFrame(results); wandb.log({"computational": wandb.Table(dataframe=df)}); return df
print("✓ Computational analysis defined")



✓ Computational analysis defined


In [25]:
# CELL 19 — MAIN TRAINING FUNCTION (REVISED)

def train_fold(fold, train_df_fold, val_df_fold, train_img_dir, device):
    print(f"\n{'='*20} FOLD {fold + 1}/{CONFIG['n_folds']} {'='*20}")

    train_dataset = ISIC2019Dataset(
        train_df_fold,
        train_img_dir,
        get_train_transforms(CONFIG["img_size"])
    )

    val_dataset = ISIC2019Dataset(
        val_df_fold,
        train_img_dir,
        get_val_transforms(CONFIG["img_size"])
    )

    train_labels = []

    for i in range(len(train_df_fold)):
        row = train_df_fold.iloc[i]
        label = 0

        for j, cls in enumerate(CONFIG["class_names"]):
            if cls in row and float(row[cls]) == 1.0:
                label = j
                break

        train_labels.append(label)

    train_labels = np.asarray(train_labels)

    sampler = create_balanced_sampler(train_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        sampler=sampler,
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    model = UHViT(
        num_classes=CONFIG["num_classes"],
        dropout_rate=CONFIG["dropout_rate"]
    ).to(device)

    samples_per_class = np.maximum(
        np.bincount(
            train_labels,
            minlength=CONFIG["num_classes"]
        ),
        1
    )

    criterion = ClassBalancedLoss(
        samples_per_class,
        CONFIG["num_classes"]
    )

    optimizer = AdamW(
        model.parameters(),
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"]
    )

    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,
        T_mult=2
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "train_bacc": [],
        "val_loss": [],
        "val_acc": [],
        "val_bacc": [],
        "val_f1": []
    }

    best_bacc = -1.0
    best_state = None

    for epoch in range(CONFIG["epochs"]):
        train_loss, train_acc, train_bacc = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scheduler,
            device,
            epoch + 1
        )

        (
            val_loss,
            val_acc,
            val_bacc,
            val_f1,
            val_preds,
            val_labels,
            val_probs
        ) = validate(
            model,
            val_loader,
            criterion,
            device
        )

        if wandb.run is not None:
            wandb.log({
                f"fold{fold}/train_loss": train_loss,
                f"fold{fold}/train_acc": train_acc,
                f"fold{fold}/train_bacc": train_bacc,
                f"fold{fold}/val_loss": val_loss,
                f"fold{fold}/val_acc": val_acc,
                f"fold{fold}/val_bacc": val_bacc,
                f"fold{fold}/val_f1": val_f1,
                "epoch": epoch + 1
            })

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_bacc"].append(train_bacc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bacc"].append(val_bacc)
        history["val_f1"].append(val_f1)

        print(
            f"E{epoch + 1:02d} "
            f"TrainBAcc={train_bacc:.4f} "
            f"ValBAcc={val_bacc:.4f} "
            f"ValF1={val_f1:.4f}"
        )

        if val_bacc > best_bacc:
            best_bacc = val_bacc
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

    if best_state is None:
        raise RuntimeError("No best model state was saved.")

    model.load_state_dict(best_state)
    model.to(device)

    checkpoint_path = (
        Path(CONFIG["checkpoint_dir"]) /
        f"fold_{fold + 1}_best.pt"
    )
    torch.save(best_state, checkpoint_path)

    (
        tta_acc,
        tta_bacc,
        tta_f1,
        tta_preds,
        tta_labels,
        tta_probs
    ) = validate_with_tta(
        model,
        val_loader,
        device
    )

    print(
        f"TTA Results: "
        f"Acc={tta_acc:.4f}, "
        f"BAcc={tta_bacc:.4f}, "
        f"F1={tta_f1:.4f}"
    )

    # One uncertainty computation only
    mean_probs_u, uncertainty_u, entropy_u, uncertainty_labels_u = (
        estimate_uncertainty(
            model,
            val_loader,
            device,
            n_samples=CONFIG["mc_dropout_samples"]
        )
    )

    plot_confusion_matrix(
        tta_labels,
        tta_preds,
        CONFIG["class_names"],
        f"{CONFIG['output_dir']}/fold_{fold + 1}_cm.png"
    )

    plot_roc_curves(
        tta_labels,
        tta_probs,
        CONFIG["class_names"],
        f"{CONFIG['output_dir']}/fold_{fold + 1}_roc.png"
    )

    plot_training_history(
        history,
        f"{CONFIG['output_dir']}/fold_{fold + 1}_history.png"
    )

    plot_uncertainty_analysis(
        uncertainty_u,
        mean_probs_u.argmax(axis=1),
        uncertainty_labels_u,
        f"{CONFIG['output_dir']}/fold_{fold + 1}_uncertainty.png"
    )

    get_gradcam_visualizations(
        model,
        val_loader,
        device,
        f"{CONFIG['output_dir']}/fold_{fold + 1}_gradcam.png"
    )

    per_class_df = per_class_analysis(
        tta_labels,
        tta_preds,
        tta_probs,
        CONFIG["class_names"]
    )

    plot_per_class_performance(
        per_class_df,
        f"{CONFIG['output_dir']}/fold_{fold + 1}_perclass.png"
    )

    return {
        "acc": tta_acc,
        "bacc": tta_bacc,
        "f1": tta_f1,
        "preds": tta_preds,
        "labels": tta_labels,
        "probs": tta_probs,
        "uncertainty": uncertainty_u,
        "entropy": entropy_u,
        "uncertainty_labels": uncertainty_labels_u,
        "model": model,
        "history": history
    }

print("Main training function defined: revised")

Main training function defined: revised


In [26]:
# Sanity check: confirm first 50 images exist
missing = 0
for i in range(min(50, len(train_df))):
    img_name = str(train_df.iloc[i]["image"])
    found = False
    for ext in [".jpg", ".jpeg", ".png", ".JPG"]:
        if (Path(train_img_dir) / f"{img_name}{ext}").exists():
            found = True
            break
    if not found:
        # try glob
        if not list(Path(train_img_dir).glob(f"{img_name}*")):
            missing += 1
            if missing <= 5:
                print("Missing example:", img_name)

print("Missing in first 50:", missing)
assert missing == 0, "Image paths are wrong — fix train_img_dir / CSV selection."

Missing in first 50: 0


In [27]:
# CELL 20: RUN 5-FOLD CROSS-VALIDATION
wandb.init(project='UHViT-SkinLesion-IF5', name='5fold-cv', config=CONFIG)
all_labels = []
for i in range(len(train_df)):
    row = train_df.iloc[i]
    for j, cls in enumerate(CONFIG['class_names']):
        if cls in row and row[cls] == 1.0: all_labels.append(j); break
    else: all_labels.append(0)
all_labels = np.array(all_labels)
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])
fold_results = []
for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, all_labels)):
    result = train_fold(fold, train_df.iloc[train_idx], train_df.iloc[val_idx], train_img_dir, CONFIG['device'])
    fold_results.append(result)
    print(f"\nFold {fold+1} Complete: BAcc={result['bacc']:.4f}, F1={result['f1']:.4f}")
accs, baccs, f1s = [r['acc'] for r in fold_results], [r['bacc'] for r in fold_results], [r['f1'] for r in fold_results]
print(f"\n{'='*50}\n5-FOLD CV RESULTS\n{'='*50}")
print(f"Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Balanced Accuracy: {np.mean(baccs):.4f} ± {np.std(baccs):.4f}")
print(f"Macro F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
acc_mean, acc_lo, acc_hi = compute_ci(accs); bacc_mean, bacc_lo, bacc_hi = compute_ci(baccs); f1_mean, f1_lo, f1_hi = compute_ci(f1s)
print(f"\n95% CI - Accuracy: [{acc_lo:.4f}, {acc_hi:.4f}]")
print(f"95% CI - Balanced Acc: [{bacc_lo:.4f}, {bacc_hi:.4f}]")
print(f"95% CI - Macro F1: [{f1_lo:.4f}, {f1_hi:.4f}]")
wandb.log({'final/acc_mean': np.mean(accs), 'final/acc_std': np.std(accs), 'final/bacc_mean': np.mean(baccs), 'final/bacc_std': np.std(baccs), 'final/f1_mean': np.mean(f1s), 'final/f1_std': np.std(f1s)})




==================== FOLD 1/5 ====================


Train E1:   0%|          | 0/634 [00:00<?, ?it/s]

Val:   0%|          | 0/159 [00:00<?, ?it/s]

E01 TrainBAcc=0.3462 ValBAcc=0.4131 ValF1=0.1156


Train E2:   0%|          | 0/634 [00:00<?, ?it/s]

...

E30 TrainBAcc=0.7563 ValBAcc=0.7396 ValF1=0.5615


TTA:   0%|          | 0/159 [00:00<?, ?it/s]

TTA Results: Acc=0.5144, BAcc=0.7441, F1=0.5727


Uncertainty T=10:   0%|          | 0/159 [00:00<?, ?it/s]


Fold 5 Complete: BAcc=0.7441, F1=0.5727

5-FOLD CV RESULTS
Accuracy: 0.5048 ± 0.0144
Balanced Accuracy: 0.7377 ± 0.0104
Macro F1: 0.5497 ± 0.0233

95% CI - Accuracy: [0.4847, 0.5248]
95% CI - Balanced Acc: [0.7233, 0.7522]
95% CI - Macro F1: [0.5173, 0.5821]


In [28]:
# NEW CELL 20D — ECE AND BRIER UTILITIES
from sklearn.metrics import brier_score_loss

def compute_ece(probs, labels, n_bins=15):
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    bounds = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_acc, bin_conf, bin_count = [], [], []
    N = len(labels)
    for i in range(n_bins):
        mask = (conf >= bounds[i]) & (conf < bounds[i + 1])
        if mask.sum() == 0:
            continue
        acc = (preds[mask] == labels[mask]).mean()
        cmean = conf[mask].mean()
        ece += mask.sum() * abs(acc - cmean) / N
        bin_acc.append(acc)
        bin_conf.append(cmean)
        bin_count.append(mask.sum())
    return ece, bin_acc, bin_conf, bin_count

def compute_brier_multiclass(probs, labels):
    C = probs.shape[1]
    total = 0.0
    for c in range(C):
        y_true = (labels == c).astype(float)
        total += brier_score_loss(y_true, probs[:, c])
    return total / C

print("ECE and Brier utilities defined")

ECE and Brier utilities defined


In [29]:
# NEW CELL 20E — T-CONVERGENCE ANALYSIS

best_model = fold_results[0]["model"]

# Reconstruct labels exactly from the ISIC one-hot columns
all_labels_arr = []

for i in range(len(train_df)):
    row = train_df.iloc[i]
    found = False

    for j, cls in enumerate(CONFIG["class_names"]):
        if cls in row and float(row[cls]) == 1.0:
            all_labels_arr.append(j)
            found = True
            break

    if not found:
        all_labels_arr.append(0)

all_labels_arr = np.asarray(all_labels_arr)

# Recreate exactly the same fold-0 split
skf_local = StratifiedKFold(
    n_splits=CONFIG["n_folds"],
    shuffle=True,
    random_state=CONFIG["seed"]
)

train_idx0, val_idx0 = list(
    skf_local.split(train_df, all_labels_arr)
)[0]

val_df0 = train_df.iloc[val_idx0].reset_index(drop=True)

val_dataset0 = ISIC2019Dataset(
    val_df0,
    train_img_dir,
    get_val_transforms(CONFIG["img_size"])
)

val_loader_fold0 = DataLoader(
    val_dataset0,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True
)

Ts = [10, 25, 50, 100]
t_results = []

for T in Ts:
    (
        mean_probs,
        uncertainty,
        entropy,
        uncertainty_labels
    ) = estimate_uncertainty(
        best_model,
        val_loader_fold0,
        CONFIG["device"],
        n_samples=T
    )

    ece_value = compute_ece(
        mean_probs,
        uncertainty_labels
    )[0]

    brier_value = compute_brier_multiclass(
        mean_probs,
        uncertainty_labels
    )

    t_results.append({
        "T": T,
        "mean_uncertainty": float(uncertainty.mean()),
        "mean_entropy": float(entropy.mean()),
        "ece": float(ece_value),
        "brier": float(brier_value)
    })

t_df = pd.DataFrame(t_results)

t_df.to_csv(
    f"{CONFIG['output_dir']}/t_convergence.csv",
    index=False
)

print(t_df.to_string(index=False))

Uncertainty T=10:   0%|          | 0/159 [00:00<?, ?it/s]

Uncertainty T=25:   0%|          | 0/159 [00:00<?, ?it/s]

Uncertainty T=50:   0%|          | 0/159 [00:00<?, ?it/s]

Uncertainty T=100:   0%|          | 0/159 [00:00<?, ?it/s]

  T  mean_uncertainty  mean_entropy      ece    brier
 10          0.023768      1.097317 0.071703 0.081897
 25          0.024252      1.099071 0.073347 0.081857
 50          0.024349      1.100339 0.068434 0.081809
100          0.024469      1.100418 0.070339 0.081836


In [30]:
# NEW CELL 20F — PLOT T-CONVERGENCE + RELIABILITY DIAGRAM
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(t_df["T"], t_df["mean_uncertainty"], "o-", label="Mean std uncertainty")
axes[0].plot(t_df["T"], t_df["mean_entropy"], "s-", label="Mean entropy")
axes[0].set_xlabel("T (MC samples)")
axes[0].set_ylabel("Uncertainty metric")
axes[0].legend()
axes[0].set_title("Uncertainty convergence vs T")

axes[1].plot(t_df["T"], t_df["ece"], "o-", label="ECE")
axes[1].plot(t_df["T"], t_df["brier"], "s-", label="Brier score")
axes[1].set_xlabel("T (MC samples)")
axes[1].set_ylabel("Calibration error")
axes[1].legend()
axes[1].set_title("Calibration vs T")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/fig_uncertainty_vs_T.png", dpi=150)
plt.close()

# Reliability diagram at T=10
mean_probs_10, unc_10, entropy_10, labels_10 = estimate_uncertainty(best_model, val_loader_fold0, CONFIG["device"], n_samples=10)
ece_10, bin_acc, bin_conf, bin_count = compute_ece(mean_probs_10, labels_10)

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.plot(bin_conf, bin_acc, "o-", label=f"MC Dropout T=10 (ECE={ece_10:.3f})")
plt.xlabel("Confidence")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Reliability diagram")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/fig_reliability_diagram.png", dpi=150)
plt.close()
print(f"ECE (T=10): {ece_10:.4f}")

Uncertainty T=10:   0%|          | 0/159 [00:00<?, ?it/s]

ECE (T=10): 0.0757


In [31]:
# NEW CELL 20G — DEEP ENSEMBLE VS MC-DROPOUT UNCERTAINTY COMPARISON

from sklearn.metrics import roc_auc_score
import copy

if "fold_results" not in globals() or len(fold_results) == 0:
    raise RuntimeError("Run CELL 20 (5-fold CV) before this cell.")

device = CONFIG["device"]
ensemble_size = 5
seeds_for_ensemble = [42, 123, 7, 99, 2026]

# Reuse the same fold-0 split as Cell 20E
all_labels_arr = []
for i in range(len(train_df)):
    row = train_df.iloc[i]
    found = False
    for j, cls in enumerate(CONFIG["class_names"]):
        if cls in row and float(row[cls]) == 1.0:
            all_labels_arr.append(j)
            found = True
            break
    if not found:
        all_labels_arr.append(0)
all_labels_arr = np.asarray(all_labels_arr)

skf_local = StratifiedKFold(
    n_splits=CONFIG["n_folds"],
    shuffle=True,
    random_state=CONFIG["seed"]
)
train_idx0, val_idx0 = list(skf_local.split(train_df, all_labels_arr))[0]

abl_train_df = train_df.iloc[train_idx0].reset_index(drop=True)
abl_val_df = train_df.iloc[val_idx0].reset_index(drop=True)

abl_train_dataset = ISIC2019Dataset(
    abl_train_df,
    train_img_dir,
    get_train_transforms(CONFIG["img_size"])
)
abl_val_dataset = ISIC2019Dataset(
    abl_val_df,
    train_img_dir,
    get_val_transforms(CONFIG["img_size"])
)

# Labels for this fold
abl_train_labels = []
for i in range(len(abl_train_df)):
    row = abl_train_df.iloc[i]
    label = 0
    for j, cls in enumerate(CONFIG["class_names"]):
        if cls in abl_train_df.columns and float(row[cls]) == 1.0:
            label = j
            break
    abl_train_labels.append(label)
abl_train_labels = np.array(abl_train_labels)

abl_sampler = create_balanced_sampler(abl_train_labels)

abl_train_loader = DataLoader(
    abl_train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=abl_sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)
abl_val_loader = DataLoader(
    abl_val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

# Train N independent models (10 epochs each, cheap budget)
ensemble_models = []
for idx, seed_val in enumerate(seeds_for_ensemble):
    print(f"\nTraining ensemble member {idx+1}/{ensemble_size} (seed={seed_val})...")
    set_seed(seed_val)
    model = UHViT(CONFIG["num_classes"], CONFIG["dropout_rate"]).to(device)
    samples_per_class = np.maximum(
        np.bincount(abl_train_labels, minlength=CONFIG["num_classes"]), 1
    )
    criterion = ClassBalancedLoss(samples_per_class, CONFIG["num_classes"])
    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

    for epoch in range(10):
        train_epoch(model, abl_train_loader, criterion, optimizer, scheduler, device, epoch + 1)

    ensemble_models.append(model)
    torch.cuda.empty_cache()

def estimate_ensemble_uncertainty(models, loader, device):
    all_means, all_stds, all_ents, all_labs = [], [], [], []

    for m in models:
        m.eval()

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Ensemble uncertainty"):
            images = images.to(device)

            probs_stack = torch.stack(
                [F.softmax(m(images), dim=1) for m in models],
                dim=0
            )

            mean_p = probs_stack.mean(0).cpu().numpy()
            std_p = probs_stack.std(0).mean(-1).cpu().numpy()
            ent_p = -(
                mean_p.clip(1e-9) *
                np.log(mean_p.clip(1e-9))
            ).sum(-1)

            all_means.append(mean_p)
            all_stds.append(std_p)
            all_ents.append(ent_p)
            all_labs.append(labels.cpu().numpy())

    return (
        np.concatenate(all_means),
        np.concatenate(all_stds),
        np.concatenate(all_ents),
        np.concatenate(all_labs)
    )
mean_probs_ens, std_ens, ent_ens, labels_ens = estimate_ensemble_uncertainty(
    ensemble_models, abl_val_loader, device
)

# MC-dropout baseline on the first ensemble member (T=10 and T=50)
base_model = ensemble_models[0]
mean_probs_mc10, unc_mc10, ent_mc10, labels_mc10 = estimate_uncertainty(
    base_model, abl_val_loader, device, n_samples=10
)
mean_probs_mc50, unc_mc50, ent_mc50, labels_mc50 = estimate_uncertainty(
    base_model, abl_val_loader, device, n_samples=50
)

# TTA-as-uncertainty: treat variance across 3 TTA views as uncertainty
def estimate_tta_uncertainty(model, loader, device):
    model.eval()
    all_means, all_stds, all_ents, all_labels = [], [], [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="TTA uncertainty"):
            images = images.to(device)
            probs_orig = F.softmax(model(images), dim=1)
            probs_h = F.softmax(model(torch.flip(images, [3])), dim=1)
            probs_v = F.softmax(model(torch.flip(images, [2])), dim=1)
            probs_stack = torch.stack([probs_orig, probs_h, probs_v], dim=0)
            mean_p = probs_stack.mean(0).cpu().numpy()
            std_p = probs_stack.std(0).mean(-1).cpu().numpy()
            ent_p = -(mean_p.clip(1e-9) * np.log(mean_p.clip(1e-9))).sum(-1)
            all_means.append(mean_p)
            all_stds.append(std_p)
            all_ents.append(ent_p)
            all_labels.append(labels.cpu().numpy())
    return np.concatenate(all_means), np.concatenate(all_stds), np.concatenate(all_ents), np.concatenate(all_labels)

mean_probs_tta, std_tta, ent_tta, labels_tta = estimate_tta_uncertainty(base_model, abl_val_loader, device)

# Uncertainty vs correctness AUROC for each method
def uncertainty_correctness_auroc(uncertainty, probs, labels):
    preds = probs.argmax(-1)
    is_wrong = (preds != labels).astype(float)
    # Higher uncertainty should predict "is_wrong"
    if len(np.unique(is_wrong)) < 2:
        return 0.5
    return roc_auc_score(is_wrong, uncertainty)

auroc_ens = uncertainty_correctness_auroc(std_ens, mean_probs_ens, labels_ens)
auroc_mc10 = uncertainty_correctness_auroc(unc_mc10, mean_probs_mc10, labels_mc10)
auroc_mc50 = uncertainty_correctness_auroc(unc_mc50, mean_probs_mc50, labels_mc50)
auroc_tta = uncertainty_correctness_auroc(std_tta, mean_probs_tta, labels_tta)

# Calibration metrics
ece_ens = compute_ece(mean_probs_ens, labels_ens)[0]
ece_mc10 = compute_ece(mean_probs_mc10, labels_mc10)[0]
ece_mc50 = compute_ece(mean_probs_mc50, labels_mc50)[0]
ece_tta = compute_ece(mean_probs_tta, labels_tta)[0]

brier_ens = compute_brier_multiclass(mean_probs_ens, labels_ens)
brier_mc10 = compute_brier_multiclass(mean_probs_mc10, labels_mc10)
brier_mc50 = compute_brier_multiclass(mean_probs_mc50, labels_mc50)
brier_tta = compute_brier_multiclass(mean_probs_tta, labels_tta)

unc_comparison_df = pd.DataFrame([
    {"method": "Deep Ensemble (N=5)", "ECE": ece_ens, "Brier": brier_ens, "mean_std_unc": float(std_ens.mean()), "mean_entropy": float(ent_ens.mean()), "uncertainty_correctness_AUROC": auroc_ens},
    {"method": "MC-Dropout T=10", "ECE": ece_mc10, "Brier": brier_mc10, "mean_std_unc": float(unc_mc10.mean()), "mean_entropy": float(ent_mc10.mean()), "uncertainty_correctness_AUROC": auroc_mc10},
    {"method": "MC-Dropout T=50", "ECE": ece_mc50, "Brier": brier_mc50, "mean_std_unc": float(unc_mc50.mean()), "mean_entropy": float(ent_mc50.mean()), "uncertainty_correctness_AUROC": auroc_mc50},
    {"method": "TTA variance (3 views)", "ECE": ece_tta, "Brier": brier_tta, "mean_std_unc": float(std_tta.mean()), "mean_entropy": float(ent_tta.mean()), "uncertainty_correctness_AUROC": auroc_tta},
])

print(unc_comparison_df.to_string(index=False))
unc_comparison_df.to_csv(f"{CONFIG['output_dir']}/uncertainty_methods_comparison.csv", index=False)

if wandb.run is not None:
    wandb.log({"uncertainty_methods_comparison": wandb.Table(dataframe=unc_comparison_df)})

# Optional: bar chart
plt.figure(figsize=(8, 5))
methods = unc_comparison_df["method"].str.replace(" (", "\n(")
plt.bar(methods, unc_comparison_df["ECE"])
plt.ylabel("ECE")
plt.title("Calibration error by uncertainty method")
plt.tight_layout()
plt.savefig(f"{CONFIG['output_dir']}/fig_uncertainty_methods_ece.png", dpi=150)
plt.close()


Training ensemble member 1/5 (seed=42)...


Train E1:   0%|          | 0/634 [00:00<?, ?it/s]

Train E2:   0%|          | 0/634 [00:00<?, ?it/s]

Train E3:   0%|          | 0/634 [00:00<?, ?it/s]

Train E4:   0%|          | 0/634 [00:00<?, ?it/s]

...

Uncertainty T=10:   0%|          | 0/159 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^Exception ignored in: ^^<function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>^
^Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
^    ^self._shutdown_workers()^
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
^    ^if w.is_alive():^
^  ^ ^ ^ ^ ^ ^^^

Uncertainty T=50:   0%|          | 0/159 [00:00<?, ?it/s]

TTA uncertainty:   0%|          | 0/159 [00:00<?, ?it/s]

                method      ECE    Brier  mean_std_unc  mean_entropy  uncertainty_correctness_AUROC
   Deep Ensemble (N=5) 0.136782 0.095072      0.055869      1.353876                       0.568604
       MC-Dropout T=10 0.256192 0.107475      0.025954      1.267379                       0.617238
       MC-Dropout T=50 0.252323 0.107414      0.026672      1.270441                       0.629633
TTA variance (3 views) 0.246705 0.106522      0.033265      1.286176                       0.575064


In [32]:
# CELL 20B — CLINICAL SAFETY: MEL operating point + ALIGNED uncertainty referral (FIXED)
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

MEL_CLASS = CONFIG["class_names"].index("MEL")

def mel_operating_point(y_true, y_probs, target_sens=0.95):
    mel_true = (y_true == MEL_CLASS).astype(int)
    mel_score = y_probs[:, MEL_CLASS]
    fpr, tpr, thr = roc_curve(mel_true, mel_score)
    idx = np.where(tpr >= target_sens)[0]
    best_idx = idx[np.argmin(fpr[idx])] if len(idx) > 0 else np.argmax(tpr)
    threshold = thr[best_idx]
    mel_pred = (mel_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(mel_true, mel_pred).ravel()
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1 = 2 * prec * sens / (prec + sens + 1e-9)
    auc = roc_auc_score(mel_true, mel_score)
    return {"target_sensitivity": target_sens, "chosen_threshold": float(threshold),
            "MEL_sensitivity": float(sens), "MEL_specificity": float(spec),
            "MEL_precision": float(prec), "MEL_F1": float(f1), "MEL_AUROC": float(auc)}

def referral_policy(y_true, y_pred, y_probs, uncertainties, reject_percentiles=(80, 90, 95)):
    results = []
    uncertainties = np.asarray(uncertainties)
    for p in reject_percentiles:
        thr = np.percentile(uncertainties, p)
        keep = uncertainties <= thr
        coverage = keep.mean()
        yt, yp = y_true[keep], y_pred[keep]
        acc = (yp == yt).mean() if len(yt) else 0
        mel_true = (yt == MEL_CLASS).astype(int)
        mel_pred = (yp == MEL_CLASS).astype(int)
        tn, fp, fn, tp = confusion_matrix(mel_true, mel_pred, labels=[0, 1]).ravel()
        mel_sens = tp / (tp + fn + 1e-9)
        results.append({"reject_percentile": p, "uncertainty_threshold": float(thr),
                         "coverage": float(coverage), "accuracy_on_kept": float(acc),
                         "MEL_sensitivity_on_kept": float(mel_sens),
                         "n_kept": int(len(yt)), "n_referred": int((~keep).sum())})
    return pd.DataFrame(results)

# Pool ALIGNED predictions + uncertainties across folds
all_y_true  = np.concatenate([r["labels"] for r in fold_results])
all_y_pred  = np.concatenate([r["preds"] for r in fold_results])
all_y_prob  = np.concatenate([r["probs"] for r in fold_results])
all_unc     = np.concatenate([r["uncertainty"] for r in fold_results])

mel_report = mel_operating_point(all_y_true, all_y_prob, target_sens=0.95)
mel_df = pd.DataFrame([mel_report])
print(mel_df.to_string(index=False))

referral_df = referral_policy(all_y_true, all_y_pred, all_y_prob, all_unc)
print(referral_df.to_string(index=False))
referral_df.to_csv(f"{CONFIG['output_dir']}/referral_policy_results.csv", index=False)

if wandb.run is not None:
    wandb.log({"clinical/MEL_operating_point": wandb.Table(dataframe=mel_df)})
    wandb.log({"clinical/referral_policy": wandb.Table(dataframe=referral_df)})

 target_sensitivity  chosen_threshold  MEL_sensitivity  MEL_specificity  MEL_precision   MEL_F1  MEL_AUROC
               0.95          0.113314         0.950022         0.309337       0.230126 0.370505   0.805837
 reject_percentile  uncertainty_threshold  coverage  accuracy_on_kept  MEL_sensitivity_on_kept  n_kept  n_referred
                80               0.028282  0.800008          0.515223                 0.619613   20265        5066
                90               0.031643  0.900004          0.510790                 0.619792   22798        2533
                95               0.034808  0.949982          0.508145                 0.617211   24064        1267


In [33]:
# NEW CELL 20B1 — UNCERTAINTY VS CORRECTNESS + NUMERICAL EXAMPLE

if not all(name in globals() for name in ["all_y_true", "all_y_pred", "all_unc"]):
    raise RuntimeError("Run CELL 20B before this cell.")

is_wrong = (all_y_pred != all_y_true).astype(float)

# Point-biserial correlation (Pearson here is fine for binary label)
from scipy.stats import pearsonr
corr, p_val = pearsonr(all_unc, is_wrong)

# AUROC: can uncertainty rank wrong predictions higher?
from sklearn.metrics import roc_auc_score
if len(np.unique(is_wrong)) == 2:
    auroc_wrong = roc_auc_score(is_wrong, all_unc)
else:
    auroc_wrong = np.nan

# Decile analysis: error rate in top vs bottom uncertainty decile
decile_boundaries = np.percentile(all_unc, np.linspace(0, 100, 11))
decile_error_rates = []
for i in range(10):
    mask = (all_unc >= decile_boundaries[i]) & (all_unc <= decile_boundaries[i+1])
    if mask.sum() == 0:
        decile_error_rates.append(np.nan)
    else:
        decile_error_rates.append(float(is_wrong[mask].mean()))

decile_df = pd.DataFrame({
    "decile": list(range(1, 11)),
    "uncertainty_range_low": decile_boundaries[:-1],
    "uncertainty_range_high": decile_boundaries[1:],
    "error_rate": decile_error_rates
})

print(f"Correlation(uncertainty, wrong prediction): r = {corr:.3f}, p = {p_val:.2e}")
print(f"AUROC(uncertainty → wrong prediction): {auroc_wrong:.3f}")
print("\nError rate by uncertainty decile:")
print(decile_df.to_string(index=False))

decile_df.to_csv(f"{CONFIG['output_dir']}/uncertainty_decile_error_rates.csv", index=False)

if wandb.run is not None:
    wandb.log({
        "uncertainty_correctness_correlation": corr,
        "uncertainty_correctness_p_value": p_val,
        "uncertainty_correctness_auroc": auroc_wrong,
        "uncertainty_decile_error_rates": wandb.Table(dataframe=decile_df)
    })

# Numerical example for the paper (Reviewer 1, point 4)
bottom_decile_error = decile_error_rates[0]
top_decile_error = decile_error_rates[-1]
print(
    f"\nNumerical example for manuscript:\n"
    f"Predictions in the lowest uncertainty decile have error rate {bottom_decile_error:.1%},\n"
    f"whereas those in the highest uncertainty decile have error rate {top_decile_error:.1%}.\n"
    f"Thus, high-uncertainty predictions are ~{top_decile_error/max(bottom_decile_error,1e-9):.1f}x more likely to be wrong."
)

Correlation(uncertainty, wrong prediction): r = 0.156, p = 2.63e-137
AUROC(uncertainty → wrong prediction): 0.580

Error rate by uncertainty decile:
 decile  uncertainty_range_low  uncertainty_range_high  error_rate
      1               0.000515                0.013444    0.270324
      2               0.013444                0.016890    0.408445
      3               0.016890                0.019055    0.484215
      4               0.019055                0.020856    0.523283
      5               0.020856                0.022506    0.527624
      6               0.022506                0.024164    0.541042
      7               0.024164                0.026017    0.559984
      8               0.026017                0.028282    0.563536
      9               0.028282                0.031643    0.524467
     10               0.031643                0.061844    0.549724

Numerical example for manuscript:
Predictions in the lowest uncertainty decile have error rate 27.0%,
whereas tho

In [34]:
# CELL 20C — MEL ADDITIONAL OPERATING POINTS

from sklearn.metrics import (
    roc_curve,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)

MEL_CLASS = CONFIG["class_names"].index("MEL")

def summarize_threshold(mel_true, mel_score, threshold):
    mel_pred = (mel_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        mel_true,
        mel_pred,
        labels=[0, 1]
    ).ravel()

    sensitivity = tp / (tp + fn + 1e-9)
    specificity = tn / (tn + fp + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    f1_value = (
        2 * precision * sensitivity /
        (precision + sensitivity + 1e-9)
    )

    return {
        "threshold": float(threshold),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "f1": float(f1_value),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn)
    }

# These arrays are created in CELL 20B
if not all(name in globals() for name in [
    "all_y_true",
    "all_y_prob"
]):
    raise RuntimeError(
        "Run CELL 20B before running CELL 20C."
    )

mel_true = (
    all_y_true == MEL_CLASS
).astype(int)

mel_score = all_y_prob[:, MEL_CLASS]

mel_auroc = roc_auc_score(
    mel_true,
    mel_score
)

mel_prauc = average_precision_score(
    mel_true,
    mel_score
)

fpr, tpr, thresholds = roc_curve(
    mel_true,
    mel_score
)

youden_j = tpr - fpr
youden_idx = int(np.argmax(youden_j))
youden_threshold = thresholds[youden_idx]

youden_report = summarize_threshold(
    mel_true,
    mel_score,
    youden_threshold
)

youden_report.update({
    "metric": "Youden-J optimal",
    "MEL_AUROC": float(mel_auroc),
    "MEL_PRAUC": float(mel_prauc),
    "Youden_J": float(youden_j[youden_idx])
})

# 90% sensitivity operating point
target_sensitivity = 0.90
candidate_idx = np.where(
    tpr >= target_sensitivity
)[0]

if len(candidate_idx) > 0:
    sensitivity_idx = candidate_idx[
        np.argmin(fpr[candidate_idx])
    ]

    sensitivity_threshold = thresholds[
        sensitivity_idx
    ]

    sensitivity_report = summarize_threshold(
        mel_true,
        mel_score,
        sensitivity_threshold
    )

    sensitivity_report.update({
        "metric": "Sensitivity-90",
        "target_sensitivity": target_sensitivity,
        "MEL_AUROC": float(mel_auroc),
        "MEL_PRAUC": float(mel_prauc)
    })
else:
    sensitivity_report = None

reports = [youden_report]

if sensitivity_report is not None:
    reports.append(sensitivity_report)

mel_additional_df = pd.DataFrame(reports)

print(
    mel_additional_df.to_string(index=False)
)

mel_additional_df.to_csv(
    f"{CONFIG['output_dir']}/mel_additional_operating_points.csv",
    index=False
)

if wandb.run is not None:
    wandb.log({
        "clinical/MEL_additional_operating_points":
            wandb.Table(dataframe=mel_additional_df),
        "clinical/MEL_AUROC": float(mel_auroc),
        "clinical/MEL_PRAUC": float(mel_prauc)
    })

 threshold  sensitivity  specificity  precision       f1   tp    fp    tn   fn           metric  MEL_AUROC  MEL_PRAUC  Youden_J  target_sensitivity
  0.341148     0.710969     0.740641   0.373316 0.489569 3215  5397 15412 1307 Youden-J optimal   0.805837   0.554059   0.45161                 NaN
  0.192535     0.900265     0.444904   0.260594 0.404190 4071 11551  9258  451   Sensitivity-90   0.805837   0.554059       NaN                 0.9


In [35]:
# NEW CELL — CLINICAL COST-BENEFIT ANALYSIS

# Illustrative costs used only for sensitivity analysis.
# These are not direct monetary clinical costs.
C_FN = 10.0   # cost of a missed melanoma
C_FP = 1.0    # cost of an unnecessary positive referral
C_REF = 0.5   # cost of specialist referral

required_arrays = [
    "all_y_true",
    "all_y_pred",
    "all_unc",
    "MEL_CLASS"
]

missing = [
    name for name in required_arrays
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}. "
        "Run CELL 20B before this cell."
    )

cost_results = []

for referral_rate in [0, 10, 20, 30, 40, 50]:
    if referral_rate == 0:
        keep = np.ones(
            len(all_unc),
            dtype=bool
        )
    else:
        uncertainty_threshold = np.percentile(
            all_unc,
            100 - referral_rate
        )

        keep = all_unc <= uncertainty_threshold

    refer = ~keep

    y_true_kept = all_y_true[keep]
    y_pred_kept = all_y_pred[keep]

    mel_true_kept = (
        y_true_kept == MEL_CLASS
    ).astype(int)

    mel_pred_kept = (
        y_pred_kept == MEL_CLASS
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        mel_true_kept,
        mel_pred_kept,
        labels=[0, 1]
    ).ravel()

    n_referred = int(refer.sum())
    n_total = len(all_y_true)

    total_cost = (
        C_FN * fn +
        C_FP * fp +
        C_REF * n_referred
    )

    cost_per_case = total_cost / max(n_total, 1)

    sensitivity = tp / (
        tp + fn + 1e-9
    )

    specificity = tn / (
        tn + fp + 1e-9
    )

    cost_results.append({
        "referral_rate_pct": float(referral_rate),
        "coverage_pct": float(100 * keep.mean()),
        "n_kept": int(keep.sum()),
        "n_referred": n_referred,
        "MEL_sensitivity_on_kept": float(sensitivity),
        "MEL_specificity_on_kept": float(specificity),
        "false_negatives_on_kept": int(fn),
        "false_positives_on_kept": int(fp),
        "total_cost": float(total_cost),
        "expected_cost_per_case": float(cost_per_case)
    })

cost_df = pd.DataFrame(cost_results)

print(cost_df.to_string(index=False))

cost_df.to_csv(
    f"{CONFIG['output_dir']}/cost_benefit_analysis.csv",
    index=False
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12, 5)
)

axes[0].plot(
    cost_df["referral_rate_pct"],
    cost_df["expected_cost_per_case"],
    "o-"
)

axes[0].set_xlabel(
    "Referral rate (%)"
)

axes[0].set_ylabel(
    "Expected cost per case"
)

axes[0].set_title(
    "Expected cost versus referral rate"
)

axes[1].plot(
    cost_df["referral_rate_pct"],
    cost_df["MEL_sensitivity_on_kept"],
    "o-",
    color="darkred"
)

axes[1].set_xlabel(
    "Referral rate (%)"
)

axes[1].set_ylabel(
    "Retained melanoma sensitivity"
)

axes[1].set_title(
    "Sensitivity versus referral rate"
)

plt.tight_layout()

plt.savefig(
    f"{CONFIG['output_dir']}/fig_cost_benefit.png",
    dpi=150
)

plt.close()

print(
    "Saved cost_benefit_analysis.csv and "
    "fig_cost_benefit.png"
)

 referral_rate_pct  coverage_pct  n_kept  n_referred  MEL_sensitivity_on_kept  MEL_specificity_on_kept  false_negatives_on_kept  false_positives_on_kept  total_cost  expected_cost_per_case
               0.0    100.000000   25331           0                 0.614772                 0.794272                     1742                     4281     21701.0                0.856697
              10.0     90.000395   22798        2533                 0.619792                 0.796597                     1606                     3778     21104.5                0.833149
              20.0     80.000790   20265        5066                 0.619613                 0.802807                     1474                     3232     20505.0                0.809482
              30.0     70.001184   17732        7599                 0.622913                 0.816875                     1310                     2611     19510.5                0.770222
              40.0     60.001579   15199       10132   

In [36]:
# CELL 21: ABLATION STUDY (FIXED LABELS + BEST-EPOCH SELECTION)

print("\n" + "="*50 + "\nABLATION STUDY\n" + "="*50)

# Define ablation models
ablation_models = {
    "UHViT (Full)": lambda: UHViT(CONFIG["num_classes"], CONFIG["dropout_rate"]),
    "Swin-T Only": lambda: SwinOnly(CONFIG["num_classes"], CONFIG["dropout_rate"]),
    "EfficientNet Only": lambda: EfficientNetOnly(CONFIG["num_classes"], CONFIG["dropout_rate"]),
    "Concat Fusion": lambda: UHViTConcatFusion(CONFIG["num_classes"], CONFIG["dropout_rate"]),
}

# Use a fixed fold split for budgeted ablation (fold 1)
train_idx, val_idx = list(skf.split(train_df, all_labels))[0]
abl_train_df, abl_val_df = train_df.iloc[train_idx].reset_index(drop=True), train_df.iloc[val_idx].reset_index(drop=True)

# Datasets & loaders (same transforms as main training)
abl_train_dataset = ISIC2019Dataset(abl_train_df, train_img_dir, get_train_transforms(CONFIG["img_size"]))
abl_val_dataset   = ISIC2019Dataset(abl_val_df,   train_img_dir, get_val_transforms(CONFIG["img_size"]))

# FIXED: build exactly one label per sample
abl_train_labels = []
for i in range(len(abl_train_df)):
    row = abl_train_df.iloc[i]
    label = 0
    for j, cls in enumerate(CONFIG["class_names"]):
        if cls in abl_train_df.columns and float(row[cls]) == 1.0:
            label = j
            break
    abl_train_labels.append(label)
abl_train_labels = np.array(abl_train_labels)

abl_sampler = create_balanced_sampler(abl_train_labels)

abl_train_loader = DataLoader(
    abl_train_dataset,
    batch_size=CONFIG["batch_size"],
    sampler=abl_sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

abl_val_loader = DataLoader(
    abl_val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

ablation_results = []

# Train each ablation model for 10 epochs (budgeted), report BEST val BAcc
for name, model_fn in ablation_models.items():
    print(f"\nTraining {name}...")
    model = model_fn().to(CONFIG["device"])

    samples_per_class = np.maximum(np.bincount(abl_train_labels, minlength=CONFIG["num_classes"]), 1)
    criterion = ClassBalancedLoss(samples_per_class, CONFIG["num_classes"])

    optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

    best_bacc, best_acc, best_f1 = -1.0, 0.0, 0.0

    for epoch in range(10):
        train_epoch(model, abl_train_loader, criterion, optimizer, scheduler, CONFIG["device"], epoch + 1)
        _, val_acc, val_bacc, val_f1, _, _, _ = validate(model, abl_val_loader, criterion, CONFIG["device"])

        if val_bacc > best_bacc:
            best_bacc, best_acc, best_f1 = val_bacc, val_acc, val_f1

    print(f"{name} (BEST): Acc={best_acc:.4f}, BAcc={best_bacc:.4f}, F1={best_f1:.4f}")

    ablation_results.append({
        "Model": name,
        "Accuracy_best": float(best_acc),
        "BalancedAcc_best": float(best_bacc),
        "MacroF1_best": float(best_f1),
        "epochs": 10,
        "protocol": "budgeted_single_fold_best_epoch"
    })

    del model
    torch.cuda.empty_cache()

abl_df = pd.DataFrame(ablation_results)
print("\nAblation Results (budgeted, single fold, best epoch):")
print(abl_df.to_string(index=False))

try:
    wandb.log({"ablation_study": wandb.Table(dataframe=abl_df)})
except Exception as e:
    print("W&B log skipped:", e)


ABLATION STUDY

Training UHViT (Full)...


Train E1:   0%|          | 0/634 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080><function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()    
self._shutdown_workers()  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers

      File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
if w.is_alive():
     if w.is_alive(): 
         ^ ^ ^ ^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
^    ^assert self._parent_pid == os.getpid(), 'can only test a child process'^
^ 
   File "/usr/lib/pyth

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Train E2:   0%|          | 0/634 [00:00<?, ?it/s]

...

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Train E10:   0%|          | 0/634 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Concat Fusion (BEST): Acc=0.5179, BAcc=0.7106, F1=0.4981

Ablation Results (budgeted, single fold, best epoch):
            Model  Accuracy_best  BalancedAcc_best  MacroF1_best  epochs                        protocol
     UHViT (Full)       0.352872          0.636321      0.385231      10 budgeted_single_fold_best_epoch
      Swin-T Only       0.427669          0.663071      0.423927      10 budgeted_single_fold_best_epoch
EfficientNet Only       0.414644          0.649028      0.388200      10 budgeted_single_fold_best_epoch
    Concat Fusion       0.517861          0.710568      0.498074      10 budgeted_single_fold_best_epoch


In [37]:
# CELL 21B-MF — MULTI-FOLD GATED VS CONCATENATION CONVERGENCE

N_CONVERGENCE_FOLDS = 3
N_CONVERGENCE_EPOCHS = 30

if "train_df" not in globals():
    raise RuntimeError("train_df is missing. Run the data-loading cells first.")

if "all_labels" not in globals():
    raise RuntimeError("all_labels is missing. Run CELL 20 first.")

fold_splits = list(skf.split(train_df, all_labels))[:N_CONVERGENCE_FOLDS]

convergence_models = {
    "Swin-T Only": lambda: SwinOnly(
        CONFIG["num_classes"],
        CONFIG["dropout_rate"]
    ),
    "EfficientNet Only": lambda: EfficientNetOnly(
        CONFIG["num_classes"],
        CONFIG["dropout_rate"]
    ),
    "Concat Fusion": lambda: UHViTConcatFusion(
        CONFIG["num_classes"],
        CONFIG["dropout_rate"]
    ),
    "Gated Fusion (UHViT)": lambda: UHViTWithGateLogging(
        CONFIG["num_classes"],
        CONFIG["dropout_rate"]
    ),
}

multi_fold_history = {
    model_name: {
        "bacc": [],
        "f1": []
    }
    for model_name in convergence_models
}

gate_by_class_all_folds = {
    class_name: []
    for class_name in CONFIG["class_names"]
}

for fold_number, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    print(f"\n{'=' * 60}")
    print(f"MULTI-FOLD CONVERGENCE: FOLD {fold_number}/{N_CONVERGENCE_FOLDS}")
    print(f"{'=' * 60}")

    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)

    train_dataset = ISIC2019Dataset(
        fold_train_df,
        train_img_dir,
        get_train_transforms(CONFIG["img_size"])
    )

    val_dataset = ISIC2019Dataset(
        fold_val_df,
        train_img_dir,
        get_val_transforms(CONFIG["img_size"])
    )

    fold_train_labels = []

    for i in range(len(fold_train_df)):
        row = fold_train_df.iloc[i]
        label = 0

        for class_index, class_name in enumerate(CONFIG["class_names"]):
            if (
                class_name in fold_train_df.columns
                and float(row[class_name]) == 1.0
            ):
                label = class_index
                break

        fold_train_labels.append(label)

    fold_train_labels = np.asarray(fold_train_labels)

    train_sampler = create_balanced_sampler(fold_train_labels)

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        sampler=train_sampler,
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=CONFIG["num_workers"],
        pin_memory=True
    )

    samples_per_class = np.maximum(
        np.bincount(
            fold_train_labels,
            minlength=CONFIG["num_classes"]
        ),
        1
    )

    for model_name, model_factory in convergence_models.items():
        print(f"\nTraining {model_name} on fold {fold_number}")

        set_seed(CONFIG["seed"] + fold_number)

        model = model_factory().to(CONFIG["device"])

        criterion = ClassBalancedLoss(
            samples_per_class,
            CONFIG["num_classes"]
        )

        optimizer = AdamW(
            model.parameters(),
            lr=CONFIG["lr"],
            weight_decay=CONFIG["weight_decay"]
        )

        scheduler = CosineAnnealingWarmRestarts(
            optimizer,
            T_0=10,
            T_mult=2
        )

        fold_bacc_history = []
        fold_f1_history = []

        for epoch in range(N_CONVERGENCE_EPOCHS):
            train_epoch(
                model,
                train_loader,
                criterion,
                optimizer,
                scheduler,
                CONFIG["device"],
                epoch + 1
            )

            (
                _,
                _,
                val_bacc,
                val_f1,
                _,
                _,
                _
            ) = validate(
                model,
                val_loader,
                criterion,
                CONFIG["device"]
            )

            fold_bacc_history.append(float(val_bacc))
            fold_f1_history.append(float(val_f1))

            print(
                f"Fold {fold_number} | "
                f"{model_name} | "
                f"Epoch {epoch + 1:02d} | "
                f"BAcc={val_bacc:.4f} | "
                f"F1={val_f1:.4f}"
            )

        multi_fold_history[model_name]["bacc"].append(
            fold_bacc_history
        )
        multi_fold_history[model_name]["f1"].append(
            fold_f1_history
        )

        if model_name == "Gated Fusion (UHViT)":
            model.eval()

            with torch.no_grad():
                for images, labels in val_loader:
                    images = images.to(CONFIG["device"])
                    _ = model(images)

                    gate_values = model.last_gate

                    for gate_value, label in zip(
                        gate_values,
                        labels.numpy()
                    ):
                        class_name = CONFIG["class_names"][int(label)]
                        gate_by_class_all_folds[class_name].append(
                            float(gate_value)
                        )

        del model
        torch.cuda.empty_cache()

# Convert fold histories to mean and standard deviation across folds
multi_fold_summary = {}

for model_name, metrics in multi_fold_history.items():
    bacc_array = np.asarray(metrics["bacc"])
    f1_array = np.asarray(metrics["f1"])

    multi_fold_summary[model_name] = {
        "bacc_mean": bacc_array.mean(axis=0).tolist(),
        "bacc_std": bacc_array.std(axis=0).tolist(),
        "f1_mean": f1_array.mean(axis=0).tolist(),
        "f1_std": f1_array.std(axis=0).tolist()
    }

gate_summary = {}

for class_name, values in gate_by_class_all_folds.items():
    gate_summary[class_name] = {
        "mean": float(np.mean(values)) if values else None,
        "std": float(np.std(values)) if values else None,
        "n": int(len(values))
    }

multi_fold_output = {
    "n_folds": N_CONVERGENCE_FOLDS,
    "n_epochs": N_CONVERGENCE_EPOCHS,
    "models": multi_fold_summary,
    "gate_by_class": gate_summary
}

with open(
    f"{CONFIG['output_dir']}/convergence_multifold_summary.json",
    "w"
) as f:
    json.dump(multi_fold_output, f, indent=2)

print(
    f"\nSaved multi-fold convergence results to "
    f"{CONFIG['output_dir']}/convergence_multifold_summary.json"
)


MULTI-FOLD CONVERGENCE: FOLD 1/3

Training Swin-T Only on fold 1


Train E1:   0%|          | 0/634 [00:00<?, ?it/s]

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080> 
 Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
       self._shutdown_workers() 
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
      if w.is_alive():^
^ ^ ^ ^ ^ ^ ^ ^^^^^^^^^^^^^^^^^

Fold 1 | Swin-T Only | Epoch 01 | BAcc=0.4617 | F1=0.1690


...

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Fold 3 | Gated Fusion (UHViT) | Epoch 29 | BAcc=0.7534 | F1=0.5963


Train E30:   0%|          | 0/634 [00:00<?, ?it/s]

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Fold 3 | Gated Fusion (UHViT) | Epoch 30 | BAcc=0.7407 | F1=0.5989

Saved multi-fold convergence results to /workspace/outputs/convergence_multifold_summary.json


In [38]:
# CELL 21C — MULTI-FOLD CONVERGENCE PLOTS

model_names = [
    "Swin-T Only",
    "EfficientNet Only",
    "Concat Fusion",
    "Gated Fusion (UHViT)"
]

fig, axes = plt.subplots(
    1,
    2,
    figsize=(15, 6)
)

epochs = np.arange(1, N_CONVERGENCE_EPOCHS + 1)

for model_name in model_names:
    summary = multi_fold_summary[model_name]

    bacc_mean = np.asarray(summary["bacc_mean"])
    bacc_std = np.asarray(summary["bacc_std"])

    f1_mean = np.asarray(summary["f1_mean"])
    f1_std = np.asarray(summary["f1_std"])

    axes[0].plot(
        epochs,
        bacc_mean,
        label=model_name
    )

    axes[0].fill_between(
        epochs,
        bacc_mean - bacc_std,
        bacc_mean + bacc_std,
        alpha=0.15
    )

    axes[1].plot(
        epochs,
        f1_mean,
        label=model_name
    )

    axes[1].fill_between(
        epochs,
        f1_mean - f1_std,
        f1_mean + f1_std,
        alpha=0.15
    )

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Balanced accuracy")
axes[0].set_title(
    "Multi-fold balanced-accuracy convergence"
)
axes[0].legend()

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
axes[1].set_title(
    "Multi-fold macro-F1 convergence"
)
axes[1].legend()

plt.tight_layout()
plt.savefig(
    f"{CONFIG['output_dir']}/fig_multifold_convergence.png",
    dpi=150
)
plt.close()

# Gate distribution across all three folds
gate_classes = list(gate_summary.keys())
gate_means = [
    gate_summary[class_name]["mean"]
    if gate_summary[class_name]["mean"] is not None
    else 0.0
    for class_name in gate_classes
]
gate_stds = [
    gate_summary[class_name]["std"]
    if gate_summary[class_name]["std"] is not None
    else 0.0
    for class_name in gate_classes
]

plt.figure(figsize=(10, 5))
plt.bar(
    gate_classes,
    gate_means,
    yerr=gate_stds,
    capsize=4
)
plt.axhline(
    0.5,
    color="gray",
    linestyle="--",
    linewidth=1
)
plt.xlabel("Lesion class")
plt.ylabel("Mean gate value")
plt.title(
    "Multi-fold learned gate values by lesion class"
)
plt.tight_layout()
plt.savefig(
    f"{CONFIG['output_dir']}/fig_multifold_gate_distribution.png",
    dpi=150
)
plt.close()

print("Saved multi-fold convergence and gate-distribution figures.")

Saved multi-fold convergence and gate-distribution figures.


In [39]:
# CELL 22: SOTA COMPARISON (FIXED: proper samples_per_class + BEST epoch)

print("\n" + "="*50 + "\nSOTA COMPARISON (budgeted single-fold)\n" + "="*50)

# This cell assumes you already ran the FIXED ablation cell (CELL 21) that created:
#   abl_train_loader, abl_val_loader, abl_train_labels
# If not, run CELL 21 first.

assert "abl_train_loader" in globals() and "abl_val_loader" in globals() and "abl_train_labels" in globals(), \
    "Run the fixed Ablation Study cell (CELL 21) first."

# Define criterion inputs correctly for this fold
samples_per_class = np.maximum(np.bincount(abl_train_labels, minlength=CONFIG["num_classes"]), 1)
criterion = ClassBalancedLoss(samples_per_class, CONFIG["num_classes"])

sota_results = []

for sota_name in SOTA_MODELS:
    print(f"\nTraining SOTA baseline: {sota_name} ...")
    try:
        model = create_sota_model(sota_name, CONFIG["num_classes"]).to(CONFIG["device"])
        optimizer = AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

        best_bacc, best_acc, best_f1 = -1.0, 0.0, 0.0

        for epoch in range(10):
            train_epoch(model, abl_train_loader, criterion, optimizer, scheduler, CONFIG["device"], epoch + 1)
            _, val_acc, val_bacc, val_f1, _, _, _ = validate(model, abl_val_loader, criterion, CONFIG["device"])

            if val_bacc > best_bacc:
                best_bacc, best_acc, best_f1 = val_bacc, val_acc, val_f1

        print(f"{sota_name} (BEST): Acc={best_acc:.4f}, BAcc={best_bacc:.4f}, F1={best_f1:.4f}")

        sota_results.append({
            "Model": sota_name,
            "Accuracy_best": float(best_acc),
            "BalancedAcc_best": float(best_bacc),
            "MacroF1_best": float(best_f1),
            "epochs": 10,
            "protocol": "budgeted_single_fold_best_epoch"
        })

        del model
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error with {sota_name}: {e}")

# Add your model result (from 5-fold CV means) as "Ours"
# These variables accs/baccs/f1s exist in your CV summary cell; if not, you can skip or hardcode.
if "accs" in globals() and "baccs" in globals() and "f1s" in globals():
    sota_results.append({
        "Model": "UHViT (Ours, 5-fold CV mean)",
        "Accuracy_best": float(np.mean(accs)),
        "BalancedAcc_best": float(np.mean(baccs)),
        "MacroF1_best": float(np.mean(f1s)),
        "epochs": CONFIG.get("epochs", 30),
        "protocol": "5-fold_CV_mean"
    })

sota_df = pd.DataFrame(sota_results).sort_values("BalancedAcc_best", ascending=False)
print("\nSOTA Comparison Results:")
print(sota_df.to_string(index=False))

try:
    wandb.log({"sota_comparison": wandb.Table(dataframe=sota_df)})
except Exception as e:
    print("W&B log skipped:", e)


SOTA COMPARISON (budgeted single-fold)

Training SOTA baseline: resnet50 ...


model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

Train E1:   0%|          | 0/634 [00:00<?, ?it/s]

Val:   0%|          | 0/159 [00:00<?, ?it/s]

...

Val:   0%|          | 0/159 [00:00<?, ?it/s]

Train E10:   0%|          | 0/634 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7e3e14f31080>^

Traceback (most recent call last):
AssertionError  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
: can only test a child process    
self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

Val:   0%|          | 0/159 [00:00<?, ?it/s]

densenet121 (BEST): Acc=0.3949, BAcc=0.6225, F1=0.3450

SOTA Comparison Results:
                       Model  Accuracy_best  BalancedAcc_best  MacroF1_best  epochs                        protocol
UHViT (Ours, 5-fold CV mean)       0.504758          0.737716      0.549704      30                  5-fold_CV_mean
swin_tiny_patch4_window7_224       0.457865          0.692736      0.469861      10 budgeted_single_fold_best_epoch
               convnext_tiny       0.450365          0.679791      0.487193      10 budgeted_single_fold_best_epoch
             efficientnet_b3       0.453918          0.659718      0.421842      10 budgeted_single_fold_best_epoch
        vit_base_patch16_224       0.458062          0.655206      0.423103      10 budgeted_single_fold_best_epoch
                 densenet121       0.394908          0.622522      0.344995      10 budgeted_single_fold_best_epoch
                    resnet50       0.138346          0.475668      0.151078      10 budgeted_single_fold_be

In [43]:
from pathlib import Path

out = Path("workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

sotadf.to_csv(
    out / "sota_comparison.csv",
    index=False
)

sotadf.to_json(
    out / "sota_comparison.json",
    orient="records",
    indent=2
)

print("SOTA comparison saved successfully.")
print("CSV:", out / "sota_comparison.csv")
print("JSON:", out / "sota_comparison.json")
print(sotadf.to_string(index=False))

SOTA comparison saved successfully.
CSV: workspace/outputs/sota_comparison.csv
JSON: workspace/outputs/sota_comparison.json
                       Model  Accuracy_best  BalancedAcc_best  MacroF1_best  epochs                        protocol
UHViT (Ours, 5-fold CV mean)       0.504758          0.737716      0.549704      30                  5-fold_CV_mean
swin_tiny_patch4_window7_224       0.457865          0.692736      0.469861      10 budgeted_single_fold_best_epoch
               convnext_tiny       0.450365          0.679791      0.487193      10 budgeted_single_fold_best_epoch
             efficientnet_b3       0.453918          0.659718      0.421842      10 budgeted_single_fold_best_epoch
        vit_base_patch16_224       0.458062          0.655206      0.423103      10 budgeted_single_fold_best_epoch
                 densenet121       0.394908          0.622522      0.344995      10 budgeted_single_fold_best_epoch
                    resnet50       0.138346          0.475668   

In [44]:
from pathlib import Path

out = Path("workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

sotadf.to_csv(out / "sota_comparison.csv", index=False)
sotadf.to_json(
    out / "sota_comparison.json",
    orient="records",
    indent=2
)

print("SOTA saved:", (out / "sota_comparison.csv").exists())

SOTA saved: True


In [45]:
# CELL 23: CROSS-DATASET VALIDATION (HAM10000)
print("\n" + "="*50 + "\nCROSS-DATASET VALIDATION (HAM10000)\n" + "="*50)

# Reuse ham_csv / ham_img_dirs computed once in CELL 5 (no re-scanning of the filesystem)
if "ham_csv" not in globals() or "ham_img_dirs" not in globals():
    raise RuntimeError("ham_csv / ham_img_dirs not found. Run CELL 5 (download+locate) first.")

ham_dataset = HAM10000Dataset(ham_csv, ham_img_dirs, get_val_transforms(CONFIG['img_size']))
ham_loader = DataLoader(ham_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])
best_model = fold_results[0]['model']; best_model.eval()
ham_preds, ham_labels, ham_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(ham_loader, desc='HAM10000 Eval'):
        images = images.to(CONFIG['device']); outputs = best_model(images)
        ham_probs.extend(F.softmax(outputs, dim=1).cpu().numpy()); ham_preds.extend(outputs.argmax(dim=1).cpu().numpy()); ham_labels.extend(labels.numpy())
ham_preds, ham_labels, ham_probs = np.array(ham_preds), np.array(ham_labels), np.array(ham_probs)
ham_acc, ham_bacc, ham_f1 = accuracy_score(ham_labels, ham_preds), balanced_accuracy_score(ham_labels, ham_preds), f1_score(ham_labels, ham_preds, average='macro')
print(f"\nHAM10000 Results:\nAccuracy: {ham_acc:.4f}\nBalanced Accuracy: {ham_bacc:.4f}\nMacro F1: {ham_f1:.4f}")
plot_confusion_matrix(ham_labels, ham_preds, CONFIG['class_names'], f"{CONFIG['output_dir']}/ham10000_cm.png")
wandb.log({'cross_dataset/ham_acc': ham_acc, 'cross_dataset/ham_bacc': ham_bacc, 'cross_dataset/ham_f1': ham_f1})



CROSS-DATASET VALIDATION (HAM10000)


HAM10000 Eval:   0%|          | 0/313 [00:00<?, ?it/s]


HAM10000 Results:
Accuracy: 0.5159
Balanced Accuracy: 0.6788
Macro F1: 0.4467


In [46]:
# CELL 23B: EXTERNAL VALIDATION (DermaMNIST from MedMNIST) - public direct download

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import medmnist
from medmnist import INFO
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score

derma_info = INFO["dermamnist"]
DermaMNIST = getattr(medmnist, derma_info["python_class"])

# DermaMNIST: 7-class dermoscopy dataset (different label set than ISIC 8-class)
# We use it as an EXTERNAL distribution test by reporting:
#   - Top-1 confidence distribution
#   - OOD-style metrics / uncertainty (optional)
#   - And/or map to MEL vs NON-MEL if you want (DermaMNIST doesn't have MEL as a direct class label).
# For IF≈5, it’s acceptable to report "cross-dataset robustness" with caveats.

class DermaMNISTWrapper(Dataset):
    def __init__(self, split="test", transform=None):
        self.data = DermaMNIST(split=split, download=True)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img, y = self.data[idx]           # img is PIL, y is int or array
        if isinstance(y, np.ndarray):
            y = int(y.squeeze())
        else:
            y = int(y)
        img = np.array(img.convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, y

# Use val transforms (resize + normalize) for compatibility
derma_test = DermaMNISTWrapper(split="test", transform=get_val_transforms(CONFIG["img_size"]))
derma_loader = DataLoader(derma_test, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])

best_model = fold_results[0]["model"]
best_model.eval()

all_conf = []
all_entropy = []

with torch.no_grad():
    for images, _ in tqdm(derma_loader, desc="DermaMNIST External Eval"):
        images = images.to(CONFIG["device"])
        probs8 = F.softmax(best_model(images), dim=1)          # ISIC 8-class outputs
        conf = probs8.max(dim=1).values.detach().cpu().numpy() # confidence of predicted ISIC class
        ent = (-probs8 * (probs8.clamp_min(1e-9)).log()).sum(dim=1).detach().cpu().numpy()

        all_conf.extend(conf.tolist())
        all_entropy.extend(ent.tolist())

all_conf = np.array(all_conf)
all_entropy = np.array(all_entropy)

report = {
    "DermaMNIST_external_n": int(len(all_conf)),
    "mean_confidence": float(all_conf.mean()),
    "median_confidence": float(np.median(all_conf)),
    "mean_entropy": float(all_entropy.mean()),
    "median_entropy": float(np.median(all_entropy)),
}

print("DermaMNIST External Robustness Report:")
for k, v in report.items():
    print(f"{k}: {v}")

wandb.run.summary["external_note"] = (
    "External dataset label space differs from ISIC2019; "
    "we report robustness metrics (confidence/entropy) under distribution shift."
)
wandb.log({"external/note": "External dataset label space differs; we report robustness metrics (confidence/entropy) under distribution shift."})

100%|██████████| 19.7M/19.7M [00:06<00:00, 3.03MB/s]


DermaMNIST External Eval:   0%|          | 0/63 [00:00<?, ?it/s]

DermaMNIST External Robustness Report:
DermaMNIST_external_n: 2005
mean_confidence: 0.5997601598055285
median_confidence: 0.5491328239440918
mean_entropy: 1.0778101860222749
median_entropy: 1.223109483718872


In [47]:
import json
from pathlib import Path
import pandas as pd

out = Path("workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

derma_report = {
    "DermaMNIST_external_n": 2005,
    "mean_confidence": 0.5997601598055285,
    "median_confidence": 0.5491328239440918,
    "mean_entropy": 1.0778101860222749,
    "median_entropy": 1.223109483718872,
}

with open(out / "dermamnist_robustness_report.json", "w") as f:
    json.dump(derma_report, f, indent=2)

pd.DataFrame([derma_report]).to_csv(
    out / "dermamnist_robustness_report.csv",
    index=False
)

print("DermaMNIST report saved.")

DermaMNIST report saved.


In [48]:
import json
import pandas as pd
from pathlib import Path

out = Path("workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

ham_report = {
    "dataset": "HAM10000",
    "n_samples": 10015,
    "Accuracy": 0.5159,
    "BalancedAccuracy": 0.6788,
    "MacroF1": 0.4467,
    "evaluation_model": "UHViT Fold 1 best checkpoint",
}

with open(out / "ham10000_results.json", "w") as f:
    json.dump(ham_report, f, indent=2)

pd.DataFrame([ham_report]).to_csv(
    out / "ham10000_results.csv",
    index=False
)

print("HAM10000 metrics saved.")
print(ham_report)

HAM10000 metrics saved.
{'dataset': 'HAM10000', 'n_samples': 10015, 'Accuracy': 0.5159, 'BalancedAccuracy': 0.6788, 'MacroF1': 0.4467, 'evaluation_model': 'UHViT Fold 1 best checkpoint'}


In [49]:
# CELL 23C: DermaMNIST W&B logging + note (append after 23B)

import pandas as pd

# 'report' should exist from CELL 23B
wandb.log({"external/dermamnist_report": wandb.Table(dataframe=pd.DataFrame([report]))})

wandb.run.summary["external_note"] = (
    "External dataset label space differs from ISIC2019; "
    "we report robustness metrics (confidence/entropy) under distribution shift."
)
wandb.log({
    "external/note": "External dataset label space differs; we report robustness metrics (confidence/entropy) under distribution shift."
})
print("✓ Logged DermaMNIST robustness report + note to W&B")

✓ Logged DermaMNIST robustness report + note to W&B


In [50]:
# CELL 24: COMPUTATIONAL ANALYSIS
print("\n" + "="*50 + "\nCOMPUTATIONAL ANALYSIS\n" + "="*50)
comp_models = {'UHViT': lambda: UHViT(CONFIG['num_classes'], CONFIG['dropout_rate']), 'Swin-T': lambda: SwinOnly(CONFIG['num_classes'], CONFIG['dropout_rate']), 'EfficientNet-B3': lambda: EfficientNetOnly(CONFIG['num_classes'], CONFIG['dropout_rate']), 'ResNet50': lambda: create_sota_model('resnet50', CONFIG['num_classes']), 'ViT-Base': lambda: create_sota_model('vit_base_patch16_224', CONFIG['num_classes'])}
comp_df = computational_analysis(comp_models, CONFIG['device']); print("\nComputational Analysis:\n" + comp_df.to_string(index=False))




COMPUTATIONAL ANALYSIS
Analyzing UHViT...
Analyzing Swin-T...
Analyzing EfficientNet-B3...
Analyzing ResNet50...
Analyzing ViT-Base...

Computational Analysis:
          Model  Params (M)  FLOPs (G)  Time (ms)
          UHViT   40.580266  10.712108  23.827622
         Swin-T   27.527042   8.760381   9.395850
EfficientNet-B3   10.711600   1.945989  12.316656
       ResNet50   23.524424   8.260809   5.662024
       ViT-Base   85.804808  24.033695   5.064659


In [52]:
import json
import pandas as pd
from pathlib import Path

out = Path("workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

computational_rows = [
    {
        "Model": "UHViT",
        "Params_M": 40.580266,
        "FLOPs_G": 10.712108,
        "Time_ms": 23.827622,
    },
    {
        "Model": "Swin-T",
        "Params_M": 27.527042,
        "FLOPs_G": 8.760381,
        "Time_ms": 9.395850,
    },
    {
        "Model": "EfficientNet-B3",
        "Params_M": 10.711600,
        "FLOPs_G": 1.945989,
        "Time_ms": 12.316656,
    },
    {
        "Model": "ResNet50",
        "Params_M": 23.524424,
        "FLOPs_G": 8.260809,
        "Time_ms": 5.662024,
    },
    {
        "Model": "ViT-Base",
        "Params_M": 85.804808,
        "FLOPs_G": 24.033695,
        "Time_ms": 5.064659,
    },
]

compdf = pd.DataFrame(computational_rows)

compdf.to_csv(out / "computational_analysis.csv", index=False)
compdf.to_json(
    out / "computational_analysis.json",
    orient="records",
    indent=2
)

print("Computational analysis saved successfully.")
print(compdf.to_string(index=False))

Computational analysis saved successfully.
          Model  Params_M   FLOPs_G   Time_ms
          UHViT 40.580266 10.712108 23.827622
         Swin-T 27.527042  8.760381  9.395850
EfficientNet-B3 10.711600  1.945989 12.316656
       ResNet50 23.524424  8.260809  5.662024
       ViT-Base 85.804808 24.033695  5.064659


In [53]:
# CELL 25: SAVE FINAL RESULTS
results_summary = {'UHViT_5Fold_CV': {'Accuracy': f"{np.mean(accs):.4f} ± {np.std(accs):.4f}", 'Balanced_Accuracy': f"{np.mean(baccs):.4f} ± {np.std(baccs):.4f}", 'Macro_F1': f"{np.mean(f1s):.4f} ± {np.std(f1s):.4f}", 'CI_95_Accuracy': f"[{acc_lo:.4f}, {acc_hi:.4f}]", 'CI_95_Balanced_Acc': f"[{bacc_lo:.4f}, {bacc_hi:.4f}]", 'CI_95_Macro_F1': f"[{f1_lo:.4f}, {f1_hi:.4f}]"}, 'Cross_Dataset_HAM10000': {'Accuracy': f"{ham_acc:.4f}", 'Balanced_Accuracy': f"{ham_bacc:.4f}", 'Macro_F1': f"{ham_f1:.4f}"}}
with open(f"{CONFIG['output_dir']}/results_summary.json", 'w') as f: json.dump(results_summary, f, indent=2)
fold_df = pd.DataFrame([{'Fold': i+1, 'Accuracy': r['acc'], 'Balanced_Acc': r['bacc'], 'Macro_F1': r['f1']} for i, r in enumerate(fold_results)])
fold_df.to_csv(f"{CONFIG['output_dir']}/fold_results.csv", index=False)
print(f"\n{'='*50}\nTRAINING COMPLETE!\n{'='*50}")
print(f"\nResults saved to: {CONFIG['output_dir']}\nCheckpoints saved to: {CONFIG['checkpoint_dir']}")
print(f"\nFinal Results:\n  Balanced Accuracy: {np.mean(baccs):.4f} ± {np.std(baccs):.4f}\n  Macro F1: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}\n  Cross-Dataset (HAM10000) BAcc: {ham_bacc:.4f}")
wandb.finish()
print("\n✓ All done! Check W&B for full logs.")



TRAINING COMPLETE!

Results saved to: /workspace/outputs
Checkpoints saved to: /workspace/checkpoints

Final Results:
  Balanced Accuracy: 0.7377 ± 0.0104
  Macro F1: 0.5497 ± 0.0233
  Cross-Dataset (HAM10000) BAcc: 0.6788


clinical/MEL_AUROC,▁
clinical/MEL_PRAUC,▁
cross_dataset/ham_acc,▁
cross_dataset/ham_bacc,▁
cross_dataset/ham_f1,▁
epoch,▁▂▂▃▄▅▅▆▆█▂▄▆▆▆█▁▁▂▃▅▅▆▇▇█▂▃▄▄▆▇█▁▁▃▄▄▅█
final/acc_mean,▁
final/acc_std,▁
final/bacc_mean,▁
final/bacc_std,▁
+40,...



✓ All done! Check W&B for full logs.


In [54]:
import json
from pathlib import Path

out = Path("/workspace/outputs")
out.mkdir(parents=True, exist_ok=True)

final_summary = {
    "UHViT_5FoldCV": {
        "BalancedAccuracy_mean": 0.7377,
        "BalancedAccuracy_std": 0.0104,
        "MacroF1_mean": 0.5497,
        "MacroF1_std": 0.0233,
        "Accuracy_mean": 0.50476,
        "Accuracy_std": 0.01442,
        "epochs": 30,
        "protocol": "5-fold cross-validation",
    },
    "HAM10000": {
        "Accuracy": 0.51593,
        "BalancedAccuracy": 0.67879,
        "MacroF1": 0.44666,
    },
    "MelanomaClinical": {
        "AUROC": 0.80584,
        "PRAUC": 0.55406,
    },
    "saved_files": [
        "sota_comparison.csv",
        "sota_comparison.json",
        "ham10000_results.csv",
        "ham10000_results.json",
        "ham10000_cm.png",
        "dermamnist_robustness_report.csv",
        "dermamnist_robustness_report.json",
        "computational_analysis.csv",
        "computational_analysis.json",
    ],
}

with open(out / "final_results_summary.json", "w") as f:
    json.dump(final_summary, f, indent=2)

print("Saved:", out / "final_results_summary.json")

Saved: /workspace/outputs/final_results_summary.json


In [55]:
from pathlib import Path

out = Path("/workspace/outputs")

for p in sorted(out.iterdir()):
    print(f"{p.name:45} {p.stat().st_size:,} bytes")

.ipynb_checkpoints                            2,000,291 bytes
convergence_multifold_summary.json            15,391 bytes
cost_benefit_analysis.csv                     813 bytes
fig_cost_benefit.png                          102,373 bytes
fig_multifold_convergence.png                 339,229 bytes
fig_multifold_gate_distribution.png           37,184 bytes
fig_reliability_diagram.png                   53,361 bytes
fig_uncertainty_methods_ece.png               32,847 bytes
fig_uncertainty_vs_T.png                      77,604 bytes
final_results_summary.json                    757 bytes
fold_1_cm.png                                 157,546 bytes
fold_1_gradcam.png                            2,252,017 bytes
fold_1_history.png                            171,075 bytes
fold_1_perclass.png                           33,471 bytes
fold_1_roc.png                                117,359 bytes
fold_1_uncertainty.png                        77,336 bytes
fold_2_cm.png                                 155,2

In [56]:
ckpt = Path("/workspace/checkpoints")

for p in sorted(ckpt.glob("*.pt")):
    print(f"{p.name:25} {p.stat().st_size:,} bytes")

fold_1_best.pt            162,907,521 bytes
fold_2_best.pt            162,907,521 bytes
fold_3_best.pt            162,907,521 bytes
fold_4_best.pt            162,907,521 bytes
fold_5_best.pt            162,907,521 bytes
